# Documentation
**Author:** Spencer Ressel

**Created:** August 16th, 2024

***

This notebook analyzes aqua-planet simulation data from the CAM6 model run by Mu-Ting Chien. 
Specifically, MJO diagnostics are computed following the specifications listed by the CLIVAR Madden-Julian Oscillation Working Group (MJOWG).
For details, see https://atmos.uw.edu/~daehyun/mjo_diagnostics/

***

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import config
import coords

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)
logger.info("Loading imports...")

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# File management
import sys
import os

# Data anaylsis
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)
from scipy.stats import t

# Plotting
import matplotlib.pyplot as plt
plt.rcParams['mathtext.fontset'] = 'dejavusans'

# Auxiliary functions
# sys.path.insert(0, "/glade/u/home/sressel/thesis-work/python/auxiliary_functions/")
from load_aquaplanet_data import load_aquaplanet_data
from processing_functions import subset_data
from auxiliary_functions.plotting_utils import bmh_colors, tick_labeller
logger.info("Imports loaded")

# Data Analysis

## Load data

In [ ]:
reload_variables = True
experiment = '4K'

variables_to_load = [
    # "Precipitation",
    # "Outgoing Longwave Radiation",
    # "Zonal Wind",
    # "Meridional Wind",
    # "Vertical Wind",
    # "Temperature",
    # "Moisture",
    # "Geopotential Height",
    # "Longwave Heating Rate",
    # "Shortwave Heating Rate",
    # "Moist Static Energy",
    # "Latent Heat Flux",
    # "Sensible Heat Flux",
    # "Moist Static Energy",
    # "Column Moist Static Energy",
    # "Column Water Vapor",
    # "Column Temperature",
    # "Column Longwave Heating",
    # "Column Shortwave Heating"
    # "Surface Pressure",
    # "Relative Humidity"
    # "Saturation Specific Humidity"
    # "Surface Longwave Flux",
    # "Surface Shortwave Flux",
    # "TOA Longwave Flux",
    # "TOA Shortwave Flux",
    # "Net Longwave Flux",
    # "Net Shortwave Flux",
    # "Zonal Moisture Advection",
    # "Meridional Moisture Advection",
    # "Vertical Moisture Advection",
    "Cloud Ice Water Content",
    "Cloud Liquid Water Content",
    # "Cloud Fraction",
]

if reload_variables or 'variables_dict' not in globals():
    variables_dict = load_aquaplanet_data(variables_to_load, experiment, slice(100, 1000))
else:
    new_variables_dict = load_aquaplanet_data(variables_to_load, experiment)
    for variable_name, variable_data in new_variables_dict.items():
        if variable_name not in variables_dict.keys():
            logger.info(f"    Adding {variable_name} to Active Variables...")
            variables_dict[variable_name] = variable_data

# logger.info("\n".join([key for key in variables_dict.keys()]))

## Process data

### Manually process data

#### Subset data

In [ ]:
print(f"{f'Calculating divergence...':<{config.SEP_WIDTH-1}}", end="")
zonal_wind_gradient = (
    (180/np.pi)
    * variables_dict['Zonal Wind'].differentiate('lon')
    / (config.EARTH_RADIUS*np.cos(np.deg2rad(variables_dict['Zonal Wind'].lat)))
)

meridional_wind_gradient = (
    (180/np.pi)
    * variables_dict['Meridional Wind'].differentiate('lat')
    / config.EARTH_RADIUS
)

variables_dict['Divergence'] = zonal_wind_gradient + meridional_wind_gradient
variables_dict['Divergence'].name = 'Divergence'
variables_dict['Divergence'].attrs['description'] = 'Divergance calculated from continuity'
variables_dict['Divergence'].attrs['units'] = r's$^{-1}$'
variables_dict['Divergence'].attrs['short_name'] = 'δ'
variables_dict['Divergence'].attrs['file_id'] = 'div'
print(rf"{'✔':>1}")

In [ ]:
logger.info("Calculate Cloud Water Paths")
logger.info("Cloud Liquid Water Path...")
variables_dict['Cloud Liquid Water Path'] = (100/9.8)*variables_dict['Cloud Liquid Water Content'].sel(plev=slice(100, 950)).integrate('plev')
variables_dict['Cloud Liquid Water Path'].name = 'Cloud Liquid Water Path'
variables_dict['Cloud Liquid Water Path'].attrs['short_name'] = 'CLWP'
variables_dict['Cloud Liquid Water Path'].attrs['file_id'] = 'clwp'
variables_dict['Cloud Liquid Water Path'].attrs['units'] = 'kg/m^2'
variables_dict['Cloud Liquid Water Path'].attrs['description'] = 'Column-integrated cloud liquid water content'

logger.info("Cloud Ice Water Path...")
variables_dict['Cloud Ice Water Path'] = (100/9.8)*variables_dict['Cloud Ice Water Content'].sel(plev=slice(100, 950)).integrate('plev')
variables_dict['Cloud Ice Water Path'].name = 'Cloud Ice Water Path'
variables_dict['Cloud Ice Water Path'].attrs['short_name'] = 'CIWP'
variables_dict['Cloud Ice Water Path'].attrs['file_id'] = 'ciwp'
variables_dict['Cloud Ice Water Path'].attrs['units'] = 'kg/m^2'
variables_dict['Cloud Ice Water Path'].attrs['description'] = 'Column-integrated cloud ice water content'
logger.info("Finished")

In [ ]:
output_directory = f"{config.DATA_DIRECTORY}/{experiment}/variables_subset"
save_subset_variables = True

variables_subset = {}

del variables_dict['Cloud Ice Water Content']
del variables_dict['Cloud Liquid Water Content']

logger.info(f"Subset Variables")
for index, (variable_name, variable_data) in enumerate(variables_dict.items()):
    logger.info(f"{f'({index+1}/{len(variables_dict)}) {variable_name}...':<{config.SEP_WIDTH-1}}")

    variables_subset[variable_name] = subset_data(
        variable_data,
        time_bounds=slice(coords.START_TIME, coords.END_TIME),
        latitude_bounds=coords.latitude_subset_bounds,
        pressure_bounds=coords.pressure_subset_bounds
    ).drop_sel(time=coords.missing_days, errors='ignore')

if save_subset_variables:
    logger.info(f"Saving subset variables")
    for index, (variable_name, variable_data) in enumerate(variables_subset.items()):
        filename = f"{variable_name.lower().replace(' ', '_')}.nc"

        logger.info(f"{f'({index+1}/{len(variables_subset)}) {variable_name}...':<{config.SEP_WIDTH-1}}")
        logger.info(f"Output Directory: {output_directory}")
        logger.info(f"Output Filename: {filename}")

        if os.path.exists(f"{output_directory}/{filename}"):
            # Prompt user for confirmation
            response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
            # response = 'y'
            if response == 'y':
                variable_data.to_netcdf(f"{output_directory}/{filename}", mode="w")  # Save the new file
                logger.info("    File overwritten")
            else:
                logger.info("    File not overwritten")
        else:
            variable_data.to_netcdf(f"{output_directory}/{filename}")  # Save the new file
else:
    logger.info(f"Not Saving")

logger.info("Finished")

In [ ]:
filename

In [ ]:
os.path.exists(f"{output_directory}/{filename}")

In [ ]:
variables_subset['Cloud Liquid Water Path'].to_netcdf(filename, mode="w")

#### Detrend data

In [ ]:
output_directory = rf"{data_directory}/{experiment}/variables_detrended"
save_detrended_variables = False

variables_detrended = {}

print(f"{f'Detrending variables':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")
for index, (variable_name, variable_data) in enumerate(variables_subset.items()):
    print(f"{f'({index+1}/{len(variables_subset)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

    # Detrend the first half
    variable_detrended_first_half = detrend_data(
        variable_data.sel(time=first_half_subset_bounds)
    )

    # Detrend the secodn half
    variable_detrended_second_half = detrend_data(
        variable_data.sel(time=second_half_subset_bounds)
    )

    # Concatenate the two halves together
    variables_detrended[variable_name] = xr.concat(
        (
            variable_detrended_first_half,
            variable_detrended_second_half
        ),
        dim='time'
    )
    print(rf"{'✔':>1}")

if save_detrended_variables:
    print(f"{'Saving detrended variables':^{config.SEP_WIDTH}}")
    print(f"{'='*config.SEP_WIDTH}")

    for index, (variable_name, variable_data) in enumerate(variables_detrended.items()):
        print(f"{f'({index+1}/{len(variables_detrended)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

        filename = f"{output_directory}/{variable_name.lower().replace(' ', '_')}.nc"
        if os.path.exists(filename):
            # Prompt user for confirmation
            response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
            if response == 'y':
                os.remove(filename)  # Delete the existing file
                variable_data.to_netcdf(filename)  # Save the new file
                print(rf"{'✔ (overwritten)':>1}")
            else:
                print(rf"{'✘ (skipped)':>1}")
        else:
            variable_data.to_netcdf(filename)  # Save the new file
            print(rf"{'✔':>1}")
else:
    print(f"{'Not saving detrended variables':<{config.SEP_WIDTH}}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

#### Time-filter data

In [ ]:
output_directory = rf"{data_directory}/{experiment}/variables_filtered"
save_filtered_variables = True

variables_filtered = {}

print(f"{f'Filtering variables':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")
for index, (variable_name, variable_data) in enumerate(variables_subset.items()):
# for index, variable_data in enumerate([variables_subset['Outgoing Longwave Radiation']]):
    variable_name = variable_data.name
    print(f"{f'({index+1}/{len(variables_subset)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

    # Filter the first half
    variable_filtered_first_half = time_filter_data(
        variable_data.sel(time=first_half_subset_bounds),
        period_bounds
    )

    # Filter the second half
    variable_filtered_second_half = time_filter_data(
        variable_data.sel(time=second_half_subset_bounds),
        period_bounds
    )

    # Concatenate halves together
    variables_filtered[variable_name] = xr.concat(
        (
            variable_filtered_first_half,
            variable_filtered_second_half
        ),
        dim='time'
    )
    print(rf"{'✔':>1}")
print(f"{'='*config.SEP_WIDTH}")

if save_filtered_variables:
    print(f"{'Saving filtered variables':^{config.SEP_WIDTH}}")
    print(f"{'='*config.SEP_WIDTH}")

    for index, (variable_name, variable_data) in enumerate(variables_filtered.items()):
        print(f"{f'({index+1}/{len(variables_filtered)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

        filename = f"{output_directory}/{variable_name.lower().replace(' ', '_')}.nc"
        if os.path.exists(filename):
            # Prompt user for confirmation
            response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
            if response == 'y':
                os.remove(filename)  # Delete the existing file
                variable_data.to_netcdf(filename)  # Save the new file
                print(rf"{'✔ (overwritten)':>1}")
            else:
                print(rf"{'✘ (skipped)':>1}")
        else:
            variable_data.to_netcdf(filename)  # Save the new file
            print(rf"{'✔':>1}")
else:
    print(f"{'Not saving filtered variables':<{config.SEP_WIDTH}}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

#### MJO-filter data

In [ ]:
output_directory = rf"{data_directory}/{experiment}/variables_mjo_filtered"
save_mjo_filtered_variables = True

assert 'variables_filtered' in globals(), "MJO-filtering requires intra-seasonally filtered data"

print(f"{'MJO filter variables':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

variables_mjo_filtered = {}

for index, (variable_name, variable_data) in enumerate(variables_filtered.items()):
# for index, variable_data in enumerate([variables_filtered['Outgoing Longwave Radiation']]):
    variable_name = variable_data.name
    print(f"{f'({index+1}/{len(variables_filtered)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

    variable_mjo_filtered_first_half = mjo_filter_data(
        variable_data.sel(time=first_half_subset_bounds),
        wavenumber_bounds=wavenumber_bounds
    ).real

    variable_mjo_filtered_second_half = mjo_filter_data(
        variable_data.sel(time=second_half_subset_bounds),
        wavenumber_bounds=wavenumber_bounds
    ).real

    variables_mjo_filtered[variable_name] = xr.concat(
        (
            variable_mjo_filtered_first_half,
            variable_mjo_filtered_second_half,
        ),
        dim='time'
    )
    print(rf"{'✔':>1}")

if save_mjo_filtered_variables:
    print(f"{'Saving MJO-filtered variables':^{config.SEP_WIDTH}}")
    print(f"{'='*config.SEP_WIDTH}")

    for index, (variable_name, variable_data) in enumerate(variables_mjo_filtered.items()):
        print(
            f"{f'({index+1}/{len(variables_mjo_filtered)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

        filename = f"{output_directory}/{variable_name.lower().replace(' ', '_')}.nc"
        if os.path.exists(filename):
            # Prompt user for confirmation
            response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
            if response == 'y':
                os.remove(filename)  # Delete the existing file
                variable_data.to_netcdf(filename)  # Save the new file
                print(rf"{'✔ (overwritten)':>1}")
            else:
                print(rf"{'✘ (skipped)':>1}")
        else:
            variable_data.to_netcdf(filename)  # Save the new file
            print(rf"{'✔':>1}")
else:
    print(f"{'Not saving MJO-filtered variables':<{config.SEP_WIDTH}}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

#### Rossby-wave filter data

In [ ]:
output_directory = rf"{data_directory}/{experiment}/variables_rossby_filtered"
save_rossby_filtered_variables = True

assert 'variables_filtered' in globals(), "rossby-filtering requires intra-seasonally filtered data"

print(f"{'Rossby filter variables':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

variables_rossby_filtered = {}

for index, (variable_name, variable_data) in enumerate(variables_filtered.items()):
# for index, variable_data in enumerate([variables_filtered['Outgoing Longwave Radiation']]):
    # variable_name = variable_data.name
    print(f"{f'({index+1}/{len(variables_filtered)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

    variable_rossby_filtered_first_half = rossby_filter_data(
        variable_data[variable_name].sel(time=first_half_subset_bounds),
        wavenumber_bounds=wavenumber_bounds
    ).real

    variable_rossby_filtered_second_half = rossby_filter_data(
        variable_data[variable_name].sel(time=second_half_subset_bounds),
        wavenumber_bounds=wavenumber_bounds
    ).real

    variables_rossby_filtered[variable_name] = xr.concat(
        (
            variable_rossby_filtered_first_half,
            variable_rossby_filtered_second_half,
        ),
        dim='time'
    )
    print(rf"{'✔':>1}")

if save_rossby_filtered_variables:
    print(f"{'Saving rossby-filtered variables':^{config.SEP_WIDTH}}")
    print(f"{'='*config.SEP_WIDTH}")

    for index, (variable_name, variable_data) in enumerate(variables_rossby_filtered.items()):
        print(
            f"{f'({index+1}/{len(variables_rossby_filtered)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

        filename = f"{output_directory}/{variable_name.lower().replace(' ', '_')}.nc"
        if os.path.exists(filename):
            # Prompt user for confirmation
            response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
            if response == 'y':
                os.remove(filename)  # Delete the existing file
                variable_data.to_netcdf(filename)  # Save the new file
                print(rf"{'✔ (overwritten)':>1}")
            else:
                print(rf"{'✘ (skipped)':>1}")
        else:
            variable_data.to_netcdf(filename)  # Save the new file
            print(rf"{'✔':>1}")
else:
    print(f"{'Not saving rossby-filtered variables':<{config.SEP_WIDTH}}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

#### Kelvin-wave filter

In [ ]:
output_directory = rf"{data_directory}/{experiment}/variables_kelvin_filtered"
save_kelvin_filtered_variables = True

assert 'variables_subset' in globals(), "kelvin-filtering requires subset data"

print(f"{'Kelvin filter variables':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

variables_kelvin_filtered = {}

characteristic_latitude = 9
max_latitude = 10

for index, (variable_name, variable_data) in enumerate(variables_subset.items()):
    print(f"{f'({index+1}/{len(variables_subset)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

    weighting_function = xr.where(np.abs(variable_data.lat) <= 10, 1, 0)
    variable_anomalies = variable_data - variable_data.mean(dim='time')
    integrand = variable_anomalies*weighting_function*np.exp(-(variable_anomalies.lat/characteristic_latitude)**2)
    projected_variable = integrand.sel(lat=slice(-10,10)).integrate('lat')

    variable_kelvin_filtered_first_half = kelvin_filter_data(
        projected_variable.sel(time=first_half_subset_bounds),
        equivalent_depth_bounds = slice(150, 25),
        period_bounds = slice(4, 2)
    )
    variable_kelvin_filtered_second_half = kelvin_filter_data(
        projected_variable.sel(time=second_half_subset_bounds),
        equivalent_depth_bounds = slice(150, 25),
        period_bounds = slice(4, 2)
    )

    variables_kelvin_filtered[variable_name] = xr.concat(
        (
            variable_kelvin_filtered_first_half,
            variable_kelvin_filtered_second_half,
        ),
        dim='time'
    )
    print(rf"{'✔':>1}")

if save_kelvin_filtered_variables:
    print(f"{'Saving kelvin-filtered variables':^{config.SEP_WIDTH}}")
    print(f"{'='*config.SEP_WIDTH}")

    for index, (variable_name, variable_data) in enumerate(variables_kelvin_filtered.items()):
        print(
            f"{f'({index+1}/{len(variables_kelvin_filtered)}) {variable_name}...':<{config.SEP_WIDTH-1}}", end="")

        filename = f"{output_directory}/{variable_name.lower().replace(' ', '_')}.nc"
        if os.path.exists(filename):
            # Prompt user for confirmation
            response = input(f"\nFile '{filename}' already exists. Overwrite? (y/n): ").strip().lower()
            if response == 'y':
                os.remove(filename)  # Delete the existing file
                variable_data.to_netcdf(filename)  # Save the new file
                print(rf"{'✔ (overwritten)':>1}")
            else:
                print(rf"{'✘ (skipped)':>1}")
        else:
            variable_data.to_netcdf(filename)  # Save the new file
            print(rf"{'✔':>1}")
else:
    print(f"{'Not saving kelvin-filtered variables':<{config.SEP_WIDTH}}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

### Load existing processed data

#### Subset data

In [ ]:
print(f"{f'Subset data':^{config.SEP_WIDTH}}")
print(f"{f'Experiment: {experiment}':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

for index, variable in enumerate(variables_to_load):
    print(f"{f'({index+1}/{len(variables_to_load)}) {variable}...':<{config.SEP_WIDTH-1}}", end="")

    variables_subset[variable] = xr.open_dataset(
        f"{data_directory}/{experiment}/variables_subset/{variable.lower().replace(' ', '_')}.nc"
    )
    print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

#### Detrended data

In [ ]:
print(f"{f'Detrended data':^{config.SEP_WIDTH}}")
print(f"{f'Experiment: {experiment}':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

variables_detrended = {}
for index, variable in enumerate(variables_to_load):
    print(f"{f'({index+1}/{len(variables_to_load)}) {variable}...':<{config.SEP_WIDTH-1}}", end="")

    variables_detrended[variable] = xr.open_dataset(
        f"{data_directory}/{experiment}/variables_detrended/{variable.lower().replace(' ', '_')}.nc"
    )
    print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

#### Filtered data

In [ ]:
print(f"{f'Filtered data':^{config.SEP_WIDTH}}")
print(f"{f'Experiment: {experiment}':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

variables_filtered = {}
for index, variable in enumerate(variables_to_load):
    print(f"{f'({index+1}/{len(variables_to_load)}) {variable}...':<{config.SEP_WIDTH-1}}", end="")

    variables_filtered[variable] = xr.open_dataset(
        f"{data_directory}/{experiment}/variables_filtered/{variable.lower().replace(' ', '_')}.nc"
    )
    print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

#### MJO-filtered data

In [ ]:
print(f"{f'mjo_filtered data':^{config.SEP_WIDTH}}")
print(f"{f'Experiment: {experiment}':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

variables_mjo_filtered = {}
for index, variable in enumerate(variables_to_load):
    print(f"{f'({index+1}/{len(variables_to_load)}) {variable}...':<{config.SEP_WIDTH-1}}", end="")

    variables_mjo_filtered[variable] = xr.open_dataset(
        f"{data_directory}/{experiment}/variables_mjo_filtered/{variable.lower().replace(' ', '_')}.nc"
    )
    print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

# Mean State Plots

## Latitude-Longitude

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

variables_to_plot = [
    variables_subset['Precipitation'],
    variables_subset['Outgoing Longwave Radiation'],
    variables_subset['Zonal Wind'].sel(plev=200),
    variables_subset['Zonal Wind'].sel(plev=850),
    variables_subset['Geopotential Height'].sel(plev=200),
    variables_subset['Geopotential Height'].sel(plev=850),
    variables_subset['Vertical Wind'].sel(plev=500),
    variables_subset['Column Temperature'],
    variables_subset['Column Water Vapor'],
    variables_subset['Column Longwave Heating'],
    variables_subset['Column Shortwave Heating'],
    variables_subset['Moist Static Energy'].sel(plev=200),
    variables_subset['Moist Static Energy'].sel(plev=850),
    variables_subset['Latent Heat Flux'],
    variables_subset['Sensible Heat Flux'],
]

print(f"Experiment SST: {experiment}")
print(f"{'='*40}")
for variable in variables_to_plot:
    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 5))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.0)

    ax = fig.add_subplot(gs[0])
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
            f"{experiment} Time-Mean"
          + f"{(' ' + (str(variable.plev.values) + '-hPa') if 'plev' in variable.coords else '')}"
          + f" {variable.name}",
            pad=15
        )

    # Add cyclic point
    cdata, clon = cutil.add_cyclic_point(
        variable.mean(dim="time"),
        coord=variable.lon,
    )

    # Plot data
    im = ax.contourf(
        clon,
        variable.lat,
        cdata,
        levels=16,
        **{k: copy.deepcopy(v) for k, v in {
            "norm": plotting_attributes[variable.name].get("norm"),
            "cmap": plotting_attributes[variable.name].get("cmap")
        }.items() if v is not None}
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label=variable.attrs['units'],
        orientation="horizontal",
    )
    cbar.ax.tick_params(labelsize=20)

    # Axis parameters
    ax.set_aspect("equal")

    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(-30, 30)
    y_ticks = np.arange(-30, 45, 15)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(tick_labeller(y_ticks, "lat"))
    ax.set_ylabel("Latitude")

    grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"}
    ax.grid(True, **grid_kwargs)

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}_time-mean"
          + f"_{variable.attrs['file_id']}"
          + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/mean-state/latitude-longitude/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

## Longitude-Height

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False
meridional_mean_region = slice(-10, 10)

variables_to_plot = [
    variables_subset['zonal wind'.title()],
    variables_subset['meridional wind'.title()],
    variables_subset['vertical wind'.title()],
    variables_subset['temperature'.title()],
    variables_subset['moisture'.title()],
    variables_subset['geopotential height'.title()],
    variables_subset['moist static energy'.title()],
    variables_subset['longwave heating rate'.title()],
    variables_subset['shortwave heating rate'.title()],
]

print(f"Experiment SST: {experiment}")
print(f"{'='*40}")
for variable in variables_to_plot:
    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 9))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.32, wspace=0.0)

    ax = fig.add_subplot(gs[0])
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
        (
            f"{experiment} Time-Mean Meridional-Mean"
          + f" ({tick_labeller([meridional_mean_region.start], 'lat')[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat')[0]})"
          + f" {variable.name.title()}"
        ),
        pad=15
    )

    # Add cyclic point
    cdata, clon = cutil.add_cyclic_point(
        variable.sel(lat=meridional_mean_region).mean(dim=['time', 'lat']).T,
        coord=variable.lon,
    )

    # Plot data
    im = ax.contourf(
        clon,
        variable.plev,
        cdata,
        levels=16,
        **{k: copy.deepcopy(v) for k, v in {
            "norm": plotting_attributes[variable.name].get("norm"),
            "cmap": plotting_attributes[variable.name].get("cmap")
        }.items() if v is not None}
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label=variable.attrs['units'],
        orientation="horizontal",
    )
    cbar.ax.tick_params(labelsize=20)

    # Axis parameters
    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(100, 950)
    ax.set_ylabel("Pressure (hPa)")
    ax.invert_yaxis()

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}_time-mean_meridional-mean-{tick_labeller([meridional_mean_region.start], 'lat', False)[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat', False)[0]}"
          + f"_{variable.attrs['file_id']}.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/mean-state/longitude-height/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

## Latitude-Height

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

variables_to_plot = [
    variables_subset['zonal wind'.title()],
    variables_subset['meridional wind'.title()],
    variables_subset['vertical wind'.title()],
    variables_subset['temperature'.title()],
    variables_subset['moisture'.title()],
    variables_subset['geopotential height'.title()],
    variables_subset['moist static energy'.title()],
    variables_subset['longwave heating rate'.title()],
    variables_subset['shortwave heating rate'.title()],
]

print(f"Experiment SST: {experiment}")
print(f"{'='*40}")
for variable in variables_to_plot:

    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 9))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05,
              right=0.95, hspace=0.32, wspace=0.0)

    ax = fig.add_subplot(gs[0])
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
        f"{experiment} Time-Mean Zonal-Mean {variable.name.title()}",
        pad=15
    )

    # Plot data
    im = ax.contourf(
        variable.lat,
        variable.plev,
        variable.mean(dim=['time', 'lon']).T,
        levels=16,
        **{k: copy.deepcopy(v) for k, v in {
            "norm": plotting_attributes[variable.name].get("norm"),
            "cmap": plotting_attributes[variable.name].get("cmap")
        }.items() if v is not None}
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label=variable.attrs['units'],
        orientation="horizontal"
    )
    cbar.ax.tick_params(labelsize=20)

    # Axis parameters
    ax.set_xlim(-30, 30)
    x_ticks = np.arange(-30, 30+10, 10)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks, "lat"))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Latitude")

    # ax.set_yscale('log')
    ax.set_ylim(100, 950)
    ax.set_ylabel("Pressure (hPa)")
    ax.invert_yaxis()

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}_time-mean_zonal-mean"
            + f"_{variable.attrs['file_id']}.png"
        )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/mean-state/latitude-height/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

# Intraseasonally-filtered mean state

## Latitude-Longitude

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

variables_to_plot = [
    variables_filtered['precipitation'],
    variables_filtered['outgoing longwave radiation'],
    variables_filtered['zonal wind'].sel(plev=200),
    variables_filtered['zonal wind'].sel(plev=850),
    variables_filtered['vertical wind'].sel(plev=500),
    variables_filtered['column temperature'],
    variables_filtered['column water vapor'],
]

print(f"Experiment SST: {experiment}")
print(f"{'='*40}")
for variable in variables_to_plot:
    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 5))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.0)

    central_longitude = 180
    proj = ccrs.PlateCarree(central_longitude=central_longitude)
    data_crs = ccrs.PlateCarree()

    ax = fig.add_subplot(gs[0], projection=proj)
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
            f"{experiment} Time-Mean"
          + f"{(' ' + (str(variable.plev.values) + '-hPa') if 'plev' in variable.coords else '')}"
          + f" {variable.name}",
            pad=15
        )

    # Add cyclic point
    cdata, clon = cutil.add_cyclic_point(
        variable.mean(dim="time"),
        coord=variable.lon,
    )

    # Plot data
    im = ax.contourf(
        clon,
        variable.lat,
        cdata,
        levels=16,
        **{k: copy.deepcopy(v) for k, v in {
                    "norm": plotting_attributes[variable.name].get("norm"),
                    "cmap": plotting_attributes[variable.name].get("cmap")
                }.items() if v is not None}
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label=variable.attrs['units'],
        orientation="horizontal",
    )
    cbar.ax.tick_params(labelsize=20)
    # cbar.set_ticks(np.arange(1, 16, 2))

    # Axis parameters
    ax.set_aspect("equal")

    ax.set_xlim(-180, 180)
    x_ticks = np.arange(-180, 180 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks, "lon"))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(-30, 30)
    y_ticks = np.arange(-30, 45, 15)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(tick_labeller(y_ticks, "lat"))
    ax.set_ylabel("Latitude")

    grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"}
    ax.grid(True, **grid_kwargs)

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}_time-mean"
          + f"_{variable.attrs['file_id']}"
          + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/mean-state/latitude-longitude/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

## Longitude-Height

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False
meridional_mean_region = slice(-10, 10)

variables_to_plot = [
    variables_filtered['zonal wind'],
    variables_filtered['meridional wind'],
    variables_filtered['vertical wind'],
    variables_filtered['temperature'],
    variables_filtered['moisture'],
]

print(f"Experiment SST: {experiment}")
print(f"{'='*40}")
for variable in variables_to_plot:
    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 9))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.32, wspace=0.0)

    ax = fig.add_subplot(gs[0])
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
        (
            f"{experiment} Time-Mean Meridional-Mean"
          + f" ({tick_labeller([meridional_mean_region.start], 'lat')[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat')[0]})"
          + f" {variable.name.title()}"
        ),
        pad=15
    )

    # Add cyclic point
    cdata, clon = cutil.add_cyclic_point(
        variable.sel(lat=meridional_mean_region).mean(dim=['time', 'lat']).T,
        coord=variable.lon,
    )

    # Plot data
    im = ax.contourf(
        clon,
        variable.plev,
        cdata,
        levels=16,
        **{k: copy.deepcopy(v) for k, v in {
            "norm": plotting_attributes[variable.name].get("norm"),
            "cmap": plotting_attributes[variable.name].get("cmap")
        }.items() if v is not None}
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label=variable.attrs['units'],
        orientation="horizontal",
    )
    cbar.ax.tick_params(labelsize=20)

    # Axis parameters
    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(100, 950)
    ax.set_ylabel("Pressure (hPa)")
    ax.invert_yaxis()

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}_time-mean_meridional-mean-{tick_labeller([meridional_mean_region.start], 'lat', False)[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat', False)[0]}"
          + f"_{variable.attrs['file_id']}.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/mean-state/longitude-height/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

# Variance

## Latitude-Longitude

### Unfiltered

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

variables_to_plot = [
    variables_subset['precipitation'.title()],
    variables_subset['outgoing longwave radiation'.title()],
    variables_subset['zonal wind'.title()].sel(plev=200),
    variables_subset['zonal wind'.title()].sel(plev=850),
    variables_subset['vertical wind'.title()].sel(plev=500),
    variables_subset['column temperature'.title()],
    variables_subset['column water vapor'.title()],
]

print(f"Experiment SST: {experiment}")
print(f"{'='*40}")
for variable in variables_to_plot:
    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 5))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.0)

    ax = fig.add_subplot(gs[0])
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
        f"Variance of {experiment} Unfiltered"
      + f"{(' ' + (str(variable.plev.values) + '-hPa') if 'plev' in variable.coords else '')}"
      + f" {variable.name}",
        pad=15
    )

    # Add cyclic point
    cdata, clon = cutil.add_cyclic_point(
        variable.var(dim="time"),
        coord=variable.lon,
    )

    # Plot data
    im = ax.contourf(
        clon,
        variable.lat,
        cdata,
        cmap='viridis',
        levels=16
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label=rf"[{variable.attrs['units']}]$^2$",
        orientation="horizontal",
    )
    cbar.ax.tick_params(labelsize=20)

    # Axis parameters
    ax.set_aspect("equal")

    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(-30, 30)
    y_ticks = np.arange(-30, 45, 15)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(tick_labeller(y_ticks, "lat"))
    ax.set_ylabel("Latitude")

    grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"}
    ax.grid(True, **grid_kwargs)

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}_unfiltered"
          + f"_{variable.attrs['file_id']}"
          + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}_variance.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/variance/latitude-longitude/unfiltered/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

### 20-100-day Filtered

In [ ]:
# Set plotting parameters
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

variables_to_plot = [
    variables_filtered['precipitation'.title()],
    variables_filtered['outgoing longwave radiation'.title()],
    variables_filtered['zonal wind'.title()].sel(plev=200),
    variables_filtered['zonal wind'.title()].sel(plev=850),
    variables_filtered['vertical wind'.title()].sel(plev=500),
    variables_filtered['column temperature'.title()],
    variables_filtered['column water vapor'.title()],
]

for variable in variables_to_plot:
    # if 'plev' not in variable.coords:
        plt.style.use("default")
        plt.rcParams.update({"font.size": 24})

        fig = plt.figure(figsize=(16, 5))
        gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
        gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.0)

        ax = fig.add_subplot(gs[0])
        cb_ax = fig.add_subplot(gs[1])

        ax.set_title(
            f"Variance of \n{experiment} "
          + f"Intraseaonally Filtered{(' ' + (str(variable.plev.values) + '-hPa') if 'plev' in variable.coords else '')}"
          + f" {variable.name}",
            pad=15
        )

        # Add cyclic point
        cdata, clon = cutil.add_cyclic_point(
            variable.var(dim="time"),
            coord=variable.lon,
        )

        # Plot data
        im = ax.contourf(
            clon,
            variable.lat,
            cdata,
            cmap='viridis',
            levels=16
        )

        # Add colorbar
        cbar = fig.colorbar(
            im,
            cax=cb_ax,
            label=rf"[{variable.attrs['units']}]$^2$",
            orientation="horizontal",
        )
        cbar.ax.tick_params(labelsize=20)
        # cbar.set_ticks(np.arange(1, 16, 2))

        # Axis parameters
        ax.set_aspect("equal")

        ax.set_xlim(0, 360)
        x_ticks = np.arange(0, 360 + 60, 60)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.set_xlabel("Longitude")

        ax.set_ylim(-30, 30)
        y_ticks = np.arange(-30, 45, 15)
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(tick_labeller(y_ticks, "lat"))
        ax.set_ylabel("Latitude")

        grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"}
        ax.grid(True, **grid_kwargs)

        if not savefig:
            plt.show()
        else:
            save_string = (
                f"{experiment}_intraseasonally-filtered"
              + f"_{variable.attrs['file_id']}"
              + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}_variance.png"
                )
            print(f"Saving plot as {save_string}")
            plt.savefig(
                f"{output_directory}/variance/latitude-longitude/intraseasonally-filtered/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )

### Variance percentage

In [ ]:
# Set plotting parameters
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

variables_variance_percentage = {}

for variable_name in variables_filtered.keys():
    variables_variance_percentage[variable_name] = 100*(
        variables_filtered[variable_name].var(dim="time")
        / variables_subset[variable_name].var(dim="time")
    )

variables_to_plot = [
    variables_variance_percentage['precipitation'.title()],
    variables_variance_percentage['outgoing longwave radiation'.title()],
    variables_variance_percentage['zonal wind'.title()].sel(plev=200),
    variables_variance_percentage['zonal wind'.title()].sel(plev=850),
    variables_variance_percentage['vertical wind'.title()].sel(plev=500),
    variables_variance_percentage['column temperature'.title()],
    variables_variance_percentage['column water vapor'.title()],

    # 100*(variables_filtered['Precipitation'].var(dim="time")/variables_subset['Precipitation'].var(dim="time")),
    # 100*(variables_filtered['outgoing longwave radiation'].var(dim="time")/variables_subset['outgoing longwave radiation'].var(dim="time")),
    # 100*(variables_filtered['zonal wind'].sel(plev=200).var(dim="time")/variables_subset['zonal wind'].sel(plev=200).var(dim="time")),
    # 100*(variables_filtered['zonal wind'].sel(plev=850).var(dim="time")/variables_subset['zonal wind'].sel(plev=850).var(dim="time")),
    # 100*(variables_filtered['vertical wind'].sel(plev=500).var(dim="time")/variables_subset['vertical wind'].sel(plev=500).var(dim="time")),
    # 100*(variables_filtered['column temperature'].var(dim="time")/variables_subset['column temperature'].var(dim="time")),
    # 100*(variables_filtered['column water vapor'].var(dim="time")/variables_subset['column water vapor'].var(dim="time")),
]

for variable in variables_to_plot:

    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 5))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.0)

    ax = fig.add_subplot(gs[0])
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
        f"Percentage Variance of {experiment}"
      + f" Intraseaonally Filtered\n to Unfiltered{(' ' + (str(variable.plev.values) + '-hPa') if 'plev' in variable.coords else '')}"
      + f" {variable.name}",
        pad=15
)

    # Add cyclic point
    cdata, clon = cutil.add_cyclic_point(
        variable,
        coord=variable.lon,
    )


    # Plot data
    im = ax.contourf(
        clon,
        variable.lat,
        cdata,
        cmap='viridis',
        levels=np.arange(0, 55, 5),
        extend='max'
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label="%",
        orientation="horizontal",
    )
    cbar.ax.tick_params(labelsize=20)
    cbar.set_ticks(np.arange(0, 60, 10))

    # Axis parameters
    ax.set_aspect("equal")

    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(-30, 30)
    y_ticks = np.arange(-30, 45, 15)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(tick_labeller(y_ticks, "lat"))
    ax.set_ylabel("Latitude")

    grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"}
    ax.grid(True, **grid_kwargs)

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}"
          + f"_{variables_subset[variable.name.lower()].attrs['file_id']}"
          + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}_variance-percentage.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/variance/latitude-longitude/percentage/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

In [ ]:
savefig=True
plt.style.use("default")
plt.rcParams.update({"font.size": 24})

variables_to_plot = [
    variables_subset['Precipitation'],
    variables_subset['Outgoing Longwave Radiation'],
    (variables_subset['Zonal Wind'], 200),
    (variables_subset['Zonal Wind'], 850),
    (variables_subset['Geopotential Height'], 200),
    (variables_subset['Geopotential Height'], 850),
    (variables_subset['Vertical Wind'], 500),
    variables_subset['Column Temperature'],
    variables_subset['Column Water Vapor'],
    variables_subset['Column Longwave Heating'],
    variables_subset['Column Shortwave Heating'],
    (variables_subset['Moist Static Energy'], 200),
    (variables_subset['Moist Static Energy'], 850),
    variables_subset['Latent Heat Flux'],
    variables_subset['Sensible Heat Flux'],
]

for variable in variables_to_plot:

    if not isinstance(variable, tuple):
        variable_data = xr.concat(
            [variable[experiment] for experiment in experiments_list],
            dim=experiments_list)
        variable_data = variable_data.rename({"concat_dim": "experiment"})
    else:
        variable_values = variable[0]
        plev = variable[1]
        variable_data = xr.concat(
            [variable_values[experiment].sel(plev=plev) for experiment in experiments_list],
            dim=experiments_list)
        variable_data = variable_data.rename({"concat_dim": "experiment"})

    print(variable_data.name)
    time_mean_variable_variance = variable_data.var(dim='time')

    fig = plt.figure(figsize=(16, 12))
    gs = GridSpec(3, 2, width_ratios=[30,1], height_ratios=[1,1,1], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.2, wspace=0.1)

    axes = []
    axes.append(fig.add_subplot(gs[0,0]))
    axes.append(fig.add_subplot(gs[1,0]))
    axes.append(fig.add_subplot(gs[2,0]))
    cb_ax = fig.add_subplot(gs[:, 1])

    for ax, experiment in zip(axes, experiments_list):
        ax.set_title(
            f"Variance of {experiment} Unfiltered"
          + f"{(' ' + (str(time_mean_variable_variance.plev.values) + '-hPa') if 'plev' in time_mean_variable_variance.coords else '')}"
          + f" {time_mean_variable_variance.name}",
            pad=15
        )

        # Add cyclic point
        cdata, clon = cutil.add_cyclic_point(
            time_mean_variable_variance.sel(experiment=experiment),
            coord=variable_data.lon,
        )

        # Plot data
        im = ax.contourf(
            clon,
            time_mean_variable_variance.lat,
            cdata,
            levels=21
        )

        # Add colorbar
        cbar = fig.colorbar(
            im,
            cax=cb_ax,
            label=rf"[{time_mean_variable_variance.attrs['units']}]$^2$",
            orientation="vertical",
        )
        cbar.ax.tick_params(labelsize=20)

        # Axis parameters
        ax.set_aspect("equal")

        ax.set_xlim(0, 360)
        x_ticks = np.arange(0, 360 + 60, 60)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.set_xlabel("Longitude")

        ax.set_ylim(-30, 30)
        y_ticks = np.arange(-30, 45, 15)
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(tick_labeller(y_ticks, "lat"))
        ax.set_ylabel("Latitude")

        grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"}
        ax.grid(True, **grid_kwargs)

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"multi-experiment_unfiltered"
          + f"_{variable.attrs['file_id']}"
          + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}_variance.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/variance/latitude-longitude/unfiltered/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

## Longitude-Height

### Unfiltered

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False
reset_vars = False
meridional_mean_region = slice(-10, 10)

print(f"Experiment SST: {experiment}")
print(f"{'='*40}")

# Reset the variable if reset_vars == True and it already exists
if reset_vars and 'variables_to_plot' in locals():
    del variables_to_plot

if not 'variables_to_plot' in locals():
    variables_to_plot = [
        variables_subset['zonal wind'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
        variables_subset['meridional wind'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
        variables_subset['vertical wind'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
        variables_subset['temperature'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
        variables_subset['moisture'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
    ]

for variable in variables_to_plot:
    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 9))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.32, wspace=0.0)

    ax = fig.add_subplot(gs[0])
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
        (
            f"{experiment} Unfiltered Meridional-Mean"
          + f" ({tick_labeller([meridional_mean_region.start], 'lat')[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat')[0]})"
          + f" {variable.name.title()} Variance"
        ),
        pad=15
    )

    # Add cyclic point
    cdata, clon = cutil.add_cyclic_point(
        variable.T,
        coord=variable.lon,
    )

    # Plot data
    im = ax.contourf(
        clon,
        variable.plev,
        cdata,
        levels=16,
        # cmap=copy.deepcopy(variable.attrs['cmap']),
        # norm=copy.deepcopy(variable.attrs['norm'])
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label=rf"[{variable.attrs['units']}]$^{2}$",
        orientation="horizontal",
    )
    cbar.ax.tick_params(labelsize=20)

    # Axis parameters
    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    # ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
    # ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(100, 950)
    ax.set_ylabel("Pressure (hPa)")
    ax.invert_yaxis()

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}_unfiltered_meridional-mean-{tick_labeller([meridional_mean_region.start], 'lat', False)[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat', False)[0]}"
          + f"_{variable.attrs['file_id']}_variance.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/variance/longitude-height/unfiltered/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

### 20-100-day Filtered

In [ ]:
xr.set_options(keep_attrs=True)
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False
reset_vars = False
meridional_mean_region = slice(-10, 10)

print(f"Experiment SST: {experiment}")
print(f"{'='*40}")

# Reset the variable if reset_vars == True and it already exists
if reset_vars and 'variables_to_plot' in locals():
    del variables_to_plot

if not 'variables_to_plot' in locals():
    variables_to_plot = [
        variables_filtered['zonal wind'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
        variables_filtered['meridional wind'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
        variables_filtered['vertical wind'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
        variables_filtered['temperature'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
        variables_filtered['moisture'].copy(deep=True).sel(lat=meridional_mean_region).mean(dim='lat').var(dim='time'),
    ]

for variable in variables_to_plot:
    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 9))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.32, wspace=0.0)

    ax = fig.add_subplot(gs[0])
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
        (
            f"{experiment} Intraseasonally-Filtered \nMeridional-Mean"
          + f" ({tick_labeller([meridional_mean_region.start], 'lat')[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat')[0]})"
          + f" {variable.name.title()} Variance"
        ),
        pad=15
    )

    # Add cyclic point
    cdata, clon = cutil.add_cyclic_point(
        variable.T,
        coord=variable.lon,
    )

    # Plot data
    im = ax.contourf(
        clon,
        variable.plev,
        cdata,
        levels=16,
        # cmap=copy.deepcopy(variable.attrs['cmap']),
        # norm=copy.deepcopy(variable.attrs['norm'])
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label=rf"[{variable.attrs['units']}]$^{2}$",
        orientation="horizontal",
    )
    cbar.ax.tick_params(labelsize=20)

    # Axis parameters
    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Longitude")

    ax.set_ylim(100, 950)
    ax.set_ylabel("Pressure (hPa)")
    ax.invert_yaxis()

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}_intraseasonally-filtered_meridional-mean-{tick_labeller([meridional_mean_region.start], 'lat', False)[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat', False)[0]}"
          + f"_{variable.attrs['file_id']}_variance.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/variance/longitude-height/intraseasonally-filtered/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

### Variance percentage

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False
reset_vars = False

print(f"Experiment SST: {experiment}")
print(f"{'='*40}")

# Reset the variable if reset_vars == True and it already exists
if reset_vars and 'variable_variance_percentage' in locals():
    del variable_variance_percentage

# If it doesn't exist, create a new dictionary and calculate the variable
if not 'variable_variance_percentage' in locals():
    print("Creating variance percentage dict")
    variable_variance_percentage = {}

    for variable_name, variable in variables_filtered.items():
        if 'plev' in variable.coords:
            print(f"Variable: {variable_name}")
            variable_variance_percentage[variable_name] = variable.copy(deep=True).isel(time=0, lat=0, drop=True)
            variable_variance_percentage[variable_name].values = (
                variables_filtered[variable_name].mean(dim='lat').var(dim='time')/variables_subset[variable_name].mean(dim='lat').var(dim='time')
            )
else:
    print("Variance percentages variable exists, using existing data...")

print("Plotting variance percentage")
for variable in variable_variance_percentage.values():

    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    fig = plt.figure(figsize=(16, 9))
    gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.32, wspace=0.0)

    ax = fig.add_subplot(gs[0])
    cb_ax = fig.add_subplot(gs[1])

    ax.set_title(
        f"Percentage Variance of {experiment}"
      + f" Intraseaonally Filtered\n to Unfiltered Zonal-Mean"
      + f" {variable.name}",
        pad=15
    )

    cdata, clon = cutil.add_cyclic_point(
        100*variable.T,
        coord=variable.lon
    )

    # Plot data
    im = ax.contourf(
        clon,
        variable.plev,
        cdata,
        cmap='viridis',
        levels=np.arange(0, 55, 5),
        extend='max'
    )

    # Add colorbar
    cbar = fig.colorbar(
        im,
        cax=cb_ax,
        label="%",
        orientation="horizontal",
    )
    cbar.ax.tick_params(labelsize=20)
    cbar.set_ticks(np.arange(0, 60, 10))

    # Axis parameters
    ax.set_xlim(0, 360)
    x_ticks = np.arange(0, 360 + 60, 60)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Latitude")

    # ax.set_yscale('log')
    ax.set_ylim(100, 950)
    ax.set_ylabel("Pressure (hPa)")
    ax.invert_yaxis()

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}_zonal-mean"
          + f"_{variable.attrs['file_id']}.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/variance/longitude-height/percentage/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

In [ ]:
window_size = 256
f_critical = {}
[f, coh] = scipy.signal.coherence(
    variables_subset['Precipitation'].sel(lat=slice(-10,10)).mean(dim='lat').sel(lon=180),
    variables_subset['Zonal Wind'].sel(lat=slice(-10,10)).mean(dim='lat').sel(lon=180, plev=850),
    fs=1,
    window='hann',
    axis=0,
    nperseg=window_size,
    noverlap=window_size//2
)

# Estimate degrees of freedom
degrees_of_freedom_numerator = 2 * len(time) / window_size
degrees_of_freedom_denominator = len(time) // 2

p_critical_list = [0.9, 0.95]
eps = len(time)/window_size
Fp_list = [scipy.stats.f.ppf(p_critical, 2, eps-2) for p_critical in p_critical_list]
chi_list = [2*Fp/(eps-2+2*Fp) for Fp in Fp_list]

In [ ]:
plt.style.use('bmh')
periods = np.array([100, 60, 30, 20, 15, 10, 5, 3, 2])
frequency_from_period = 1 / periods
[fig, ax] = plt.subplots(1, 1, figsize=(16,9))

ax.plot(f, coh)
ax.set_aspect('auto')
ax.set_xscale('log')
ax.set_xlim(1 / 167, 1 / 2)
ax.set_ylim(0, 1)
ax.axvline(x=1/100, c='k', ls=':', zorder=-10)
ax.axvline(x=1/20, c='k', ls=':', zorder=-10)
[ax.axhline(y=chi, color='k', ls='--') for chi in chi_list]

ax.set_xticks(ticks=frequency_from_period)
ax.xaxis.set_major_formatter(mticker.FixedFormatter(periods))
ax.set_xlabel("Period (days)")
ax.set_ylabel(r'Coherence$^{2}$')

for edge in ["top", "bottom", "left", "right"]:
    ax.spines[edge].set_linewidth(4)
ax.tick_params(
    which="major",
    axis="both",
    direction="in",
    top=True,
    right=True,
    width=4,
    length=15,
    color="#bcbcbc",
    pad=10,
)

plt.show()

# Time Series Power Spectra

## Calculate power spectra

In [ ]:
variable_frequency = {}
variable_power_spectrum = {}

window_size = 256
segment_length_degrees = 20
segment_length = int(segment_length_degrees * (1/2.5))
overlap = 0

weights = np.cos(np.deg2rad(latitude))
weights.name = "weights"

variables_to_plot = [
    variables_subset['precipitation'],
    variables_subset['outgoing longwave radiation'],
    variables_subset['zonal wind'].sel(plev=200),
    variables_subset['zonal wind'].sel(plev=850),
    variables_subset['vertical wind'].sel(plev=500),
    variables_subset['column temperature'],
    variables_subset['column water vapor'],
]

print("Calculating power spectra...")
print(f"{'='*40}")
# for variable in variables_subset:
for variable in variables_to_plot:
    variable_id = f"{variable.attrs['file_id']}{(str(variable.plev.values) if 'plev' in variable.coords else '')}"
    print(f"----> {variable_id}")

    horizontal_mean_time_series = variable.sel(lat=slice(-10,10)).weighted(weights).mean(dim='lat').rolling(
        lon=segment_length, center=False
    ).construct("index")[:, (segment_length-1):][:, ::(segment_length-overlap)].transpose("time", "lon", "index").mean(dim="index")

    detrended_time_series = signal.detrend(horizontal_mean_time_series, type="linear", axis=0)

    [time_series_frequency, time_series_power_spectrum] = signal.welch(
        detrended_time_series,
        nperseg=window_size,
        noverlap=window_size // 2,
        fs=1,
        axis=0
    )

    variable_power_spectrum[variable_id] = xr.DataArray(
        data=np.mean(time_series_power_spectrum, axis=1),
        dims=["frequency"],
        coords=dict(
            frequency=time_series_frequency
        ),
        attrs=dict(
            plev=(variable.plev.values if 'plev' in variable.coords else '')
        )
    )
    variable_power_spectrum[variable_id].name = variable.name
    variable_power_spectrum[variable_id].attrs['units'] = variable.attrs['units']

    time_series_variance = np.var(detrended_time_series)
    integrated_variance = scipy.integrate.simpson(
        variable_power_spectrum[variable_id],
        x=variable_power_spectrum[variable_id].frequency
    )

    print(f"Time Series Variance:               {time_series_variance:>3.3f}")
    print(f"Power Spectrum Integrated Variance: {integrated_variance:>3.3f}")
    print(f"Percent difference:                 {100*(time_series_variance-integrated_variance)/time_series_variance:>3.1f}%")
    print(f"{'='*20}")

print(f"{'='*40}")
print("Power spectra calculated")

### Fit Red Spectrum

In [ ]:
def calculate_red_spectrum(frequency, autocorrelation):
    red_spectrum = (1 - autocorrelation**2) / (
        1 - (2 * autocorrelation * np.cos(frequency * 2 * np.pi)) + autocorrelation**2
    )
    return red_spectrum

In [ ]:
# Set plotting parameters
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = True

p_critical_list = [0.9, 0.95]

linestyle = {}
linestyle[0.9] = '--'
linestyle[0.95] = ':'

plt.style.use("bmh")
plt.rcParams.update({"font.size": 18})

periods = np.array([100, 60, 30, 20, 15, 10, 5, 3, 2])
frequency_from_period = 1 / periods

for variable_id, variable in variable_power_spectrum.items():
    print(variable_id)

    total_variance = scipy.integrate.simpson(variable_power_spectrum[variable_id], x=variable_power_spectrum[variable_id].frequency)

    red_noise_parameters = {}
    variable_covariance = {}
    variable_red_spectrum = {}

    f_critical = {}

    [red_noise_parameters[variable_id], variable_covariance[variable_id]] = curve_fit(
            calculate_red_spectrum,
            variable_power_spectrum[variable_id].frequency,
            variable_power_spectrum[variable_id],
            p0=(0.5),
        )
    variable_red_spectrum[variable_id] = calculate_red_spectrum(
        variable_power_spectrum[variable_id].frequency,
        red_noise_parameters[variable_id][0]
    )

    red_variance = scipy.integrate.simpson(variable_red_spectrum[variable_id], x=variable_power_spectrum[variable_id].frequency)

    # Estimate degrees of freedom
    degrees_of_freedom_numerator = 2 * len(time) / window_size
    degrees_of_freedom_denominator = len(time) // 2

    for p_critical in p_critical_list:
        f_critical[p_critical] = scipy.stats.f.ppf(
            p_critical, degrees_of_freedom_numerator, degrees_of_freedom_denominator
        )

    [fig, ax] = plt.subplots(1, 1, figsize=(16, 6))

    ax.set_title(
        f"{experiment} "
        + f"{((str(variable.attrs['plev']) + '-hPa ') if variable.attrs['plev'] != '' else '')}"
        + f"{variable.name} Time Series Power Spectrum",
        pad=10
    )

    ax.plot(
        variable_power_spectrum[variable_id].frequency,
        variable_power_spectrum[variable_id].frequency*variable_power_spectrum[variable_id],
        lw=3
    )

    ax.plot(
        variable_power_spectrum[variable_id].frequency,
        (total_variance/red_variance)*variable_power_spectrum[variable_id].frequency*variable_red_spectrum[variable_id],
        color=bmh_colors('red'),
        lw=3,
        label='Red Spectrum'
    )

    for p_critical in p_critical_list:
        ax.plot(
            variable_power_spectrum[variable_id].frequency,
            f_critical[p_critical] * (total_variance/red_variance)*variable_power_spectrum[variable_id].frequency*variable_red_spectrum[variable_id],
            # lw=4,
            color=bmh_colors("red"),
            ls=linestyle[p_critical],
            label=f"{100*p_critical}%",
            lw=3
        )

    ax.legend()
    ax.set_aspect('auto')
    ax.set_xscale('log')
    ax.set_xlim(1 / 167, 1 / 2)
    ax.set_ylim(bottom=0)
    ax.axvline(x=1/100, c='k', ls=':', zorder=-10)
    ax.axvline(x=1/20, c='k', ls=':', zorder=-10)
    ax.set_ylabel(rf"[{variable.attrs['units']}]$^2$")
    ax.set_xticks(ticks=frequency_from_period)
    ax.xaxis.set_major_formatter(mticker.FixedFormatter(periods))
    ax.set_xlabel("Period (days)")

    for edge in ["top", "bottom", "left", "right"]:
        ax.spines[edge].set_linewidth(4)
    ax.tick_params(
        which="major",
        axis="both",
        direction="in",
        top=True,
        right=True,
        width=4,
        length=15,
        color="#bcbcbc",
        pad=10,
    )

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}"
          + f"_{variable_id}"
         # + f"_{variable.attrs['file_id']}"
          # + f"{((str(variable.plev.values)) if 'plev' in variables_subset[variable.name.lower()].coords else '')}"
          + f"_time-series-power-spectrum.png"

            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/time-series-power-spectra/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

In [ ]:
p_critical_list = [0.9, 0.95]

red_noise_parameters = {}
variable_covariance = {}
variable_red_spectrum = {}

f_critical = {}

# for variable in variables_to_plot:
for variable_id, variable in variable_power_spectrum.items():
    [red_noise_parameters[variable_id], variable_covariance[variable_id]] = curve_fit(
        calculate_red_spectrum,
        variable.frequency,
        variable_power_spectrum[variable_id] / np.mean(variable_power_spectrum[variable_id]),
        p0=(0.5),
    )
    variable_red_spectrum[variable_id] = calculate_red_spectrum(
        variable.frequency, red_noise_parameters[variable_id][0]
    )

# Estimate degrees of freedom
degrees_of_freedom_numerator = 2 * len(time) / window_size
degrees_of_freedom_denominator = len(time) // 2

for p_critical in p_critical_list:
    f_critical[p_critical] = scipy.stats.f.ppf(
        p_critical, degrees_of_freedom_numerator, degrees_of_freedom_denominator
    )

## Plot power spectra

In [ ]:
# # Set plotting parameters
# output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
# savefig = False

# periods = np.array([100, 60, 30, 20, 15, 10, 5, 3, 2])
# frequency_from_period = 1 / periods

# plt.style.use("bmh")
# plt.rcParams.update({"font.size": 18})

# linestyle = {}
# linestyle[0.9] = '--'
# linestyle[0.95] = ':'

# print(f"Experiment SST: {experiment}")
# print(f"{'='*40}")

# for variable_id, variable in variable_power_spectrum.items():
#     [fig, ax] = plt.subplots(1, 1, figsize=(16, 6))

#     ax.set_title(
#         f"{experiment} "
#       + f"{((str(variable.attrs['plev']) + '-hPa ') if variable.attrs['plev'] != '' else '')}"
#       # + f"{variable.attrs['plev']}"
#       + f"{variable.name} Time Series Power Spectrum",
#         pad=10
#     )

#     ax.plot(
#         variable.frequency,
#         variable_power_spectrum[variable_id] / np.mean(variable_power_spectrum[variable_id]),
#         color=bmh_colors("blue"),
#         lw=4,
#     )

#     ax.plot(
#         variable.frequency,
#         variable_red_spectrum[variable_id],
#         lw=4,
#         color=bmh_colors("red"),
#         ls="-",
#         label='Red Spectrum'
#     )

#     for p_critical in p_critical_list:
#         ax.plot(
#             variable.frequency,
#             f_critical[p_critical] * variable_red_spectrum[variable_id],
#             lw=4,
#             color=bmh_colors("red"),
#             ls=linestyle[p_critical],
#             label=f"{100*p_critical}%"
#         )

#     # Add vertical lines to show the intraseasonal band
#     ax.axvline(x=1 / INTRASEASONAL_LOWCUT, ls=":", lw=4, color="black", alpha=0.5)
#     ax.axvline(x=1 / INTRASEASONAL_HIGHCUT, ls=":", lw=4, color="black", alpha=0.5)

#     # Add legend
#     ax.legend()
#     # Configure axes
#     ax.set_ylabel("power")

#     for edge in ["top", "bottom", "left", "right"]:
#         ax.spines[edge].set_linewidth(4)
#     ax.tick_params(
#         which="major",
#         axis="both",
#         direction="in",
#         top=True,
#         right=True,
#         width=4,
#         length=15,
#         color="#bcbcbc",
#         pad=10,
#     )

#     ax.set_aspect("auto")
#     ax.set_xscale("log")
#     ax.set_xlim(1 / 167, 1 / 2)
#     # ax.set_ylim(-0.1, 1.0)
#     ax.set_xticks(ticks=frequency_from_period)
#     ax.xaxis.set_major_formatter(mticker.FixedFormatter(periods))
#     ax.set_xlabel("Period (days)")

#     if not savefig:
#         plt.show()
#     else:
#         save_string = (
#             f"{experiment}"
#           + f"_{variable_id}"
#          # + f"_{variable.attrs['file_id']}"
#           # + f"{((str(variable.plev.values)) if 'plev' in variables_subset[variable.name.lower()].coords else '')}"
#           + f"_time-series-power-spectrum.png"

#             )
#         print(f"Saving plot as {save_string}")
#         plt.savefig(
#             f"{output_directory}/time-series-power-spectra/{save_string}",
#             dpi=500,
#             bbox_inches="tight",
#         )

# print(f"{'='*40}")
# print("Finished")

# EOF analysis

## Full horizontal fields

### Calculate EOFs

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False
calculate_eofs = True
plot_eofs = True

compositing_variable = variables_filtered['Precipitation'].copy(deep=True)

print(f"Variable: {compositing_variable.name}")

if calculate_eofs:
    print("Reshaping variables...")
    reshaped_variable = np.reshape(
        compositing_variable.values,
        (
            compositing_variable.shape[0],
            (
                compositing_variable.shape[1]
                * compositing_variable.shape[2]
            ),
        ),
    )
    print("Variables reshaped")

    #### Calculate EOFs and PCs
    print("Computing EOFs...")
    U, S, VT = np.linalg.svd(reshaped_variable.T, full_matrices=False)
    EOF = U.T
    PC = np.dot(np.diag(S), VT)
    print("EOFs computed")

    # Calculate nominal degrees of freedom
    nominal_degrees_of_freedom = np.size(reshaped_variable, 1)

    # Calculate eigenvalues and spectrum
    eigenvalues = S**2 / nominal_degrees_of_freedom
    eigenvalue_spectrum = eigenvalues / np.sum(eigenvalues)
    explained_variance = 100 * eigenvalue_spectrum

    # Estimate 1-lag autocorrelation and effective degrees of freedom
    lag = 1
    B = 0
    for k in range(lag - 1, nominal_degrees_of_freedom - lag):
        B = B + np.sum(reshaped_variable[:, k] * reshaped_variable[:, k + lag])
    phi_L = 1 / (nominal_degrees_of_freedom - 2 * lag) * B
    phi_0 = 1 / nominal_degrees_of_freedom * np.sum(reshaped_variable**2)
    autocorrelation = phi_L / phi_0
    degrees_of_freedom = (
        (1 - autocorrelation**2) / (1 + autocorrelation**2)
    ) * nominal_degrees_of_freedom

    # Estimate uncertainty in eigenvalue spectrum
    spectrum_error = eigenvalue_spectrum * np.sqrt(2 / degrees_of_freedom)

else:
    print("Using pre-calculated EOFs")

if plot_eofs:
    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})
    cmap_modified = mjo.modified_colormap("coolwarm", "white", 0.1, 0.1)

    fig = plt.figure(figsize=(16, 16))
    gs = GridSpec(4, 2, width_ratios=[1, 0.02], figure=fig)
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.05, wspace=0.1)

    fig.suptitle(f"EOFs 1-4 of {experiment} "
                 + f"{((str(compositing_variable.plev.values) + '-hPa ') if 'plev' in compositing_variable.coords else '')}"
                 +f"{compositing_variable.name}", x=0.5, y=0.975, ha='center')

    axes = []
    axes.append(fig.add_subplot(gs[0, 0]))
    axes.append(fig.add_subplot(gs[1, 0]))
    axes.append(fig.add_subplot(gs[2, 0]))
    axes.append(fig.add_subplot(gs[3, 0]))

    cb_ax = fig.add_subplot(gs[0:4, 1])

    nth = {1: "1st", 2: "2nd", 3: "3rd", 4: "4th"}

    fig_labels = {1: "a)", 2: "b)", 3: "c)", 4: "d)"}

    mult = [1, 1, 1, 1]

    for index, ax in enumerate(axes):
        ax.set_title(
            f"{fig_labels[index+1]} {nth[index+1]} mode ({100*eigenvalue_spectrum[index]:0.2f}%)",
            loc="left",
        )

        # Add cyclic point
        cdata, clon = cutil.add_cyclic_point(
            np.std(PC[index])
            * np.reshape(
                EOF[index],
                (
                    compositing_variable.shape[1],
                    compositing_variable.shape[2],
                ),
            ),
            coord=compositing_variable.lon,
        )

        # Plot data
        im = ax.contourf(
            clon,
            compositing_variable.lat,
            cdata,
            cmap=cmap_modified,
            levels=31,
            norm=mcolors.CenteredNorm(vcenter=0),
        )

        # Plot data
        ax.contour(
            clon,
            compositing_variable.lat,
            cdata,
            colors="k",
            levels=im.levels[np.abs(im.levels) >= np.quantile(np.abs(im.levels), 0.01)][::4],
            linewidths=1,
        )

    # Add colorbar
    cbar = fig.colorbar(im, cax=cb_ax)
    cbar.ax.tick_params(labelsize=20)
    cbar.locator = mticker.MaxNLocator(nbins=15)
    cbar.set_label(f"{compositing_variable.attrs['units']}")

    for ax in axes:
        ax.set_aspect("equal")
        ax.grid(True, **grid_kwargs)

        ax.set_xlim(0, 360)
        x_ticks = np.arange(0, 360 + 60, 60)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.set_xlabel("Longitude")

        ax.set_ylim(-30, 30)
        ax.set_yticks(np.arange(-30, 45, 15))
        ax.set_yticklabels(tick_labeller(np.arange(-30, 45, 15), 'lat'))
        ax.set_ylabel("Latitude")

        for spine in ax.spines.values():
            spine.set_edgecolor("black")
            spine.set_linewidth(2)

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}"
          + f"_{compositing_variable.attrs['file_id']}"
          + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}_EOF-maps.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/EOFs/single-variable-latitude-longitude/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

### Eigenvalue Spectra

In [ ]:
output_directory = "/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

print(f"Variable: {compositing_variable.name}")

plt.style.use("bmh")
plt.rcParams.update({"font.size": 24})

[fig, ax] = plt.subplots(figsize=(16, 9))
num_modes = 10
EOF_index = np.arange(1, num_modes + 1, 1)

ax.set_title(
    f"Eigenvalue spectrum of {experiment} "
  + f"{((str(compositing_variable.plev.values) + '-hPa ') if 'plev' in compositing_variable.coords else '')}"
  + f"{compositing_variable.name} EOFs",
    pad=10
)
ax.bar(EOF_index, 100 * eigenvalue_spectrum[:num_modes], width=0.5, color="dodgerblue")

ax.errorbar(
    EOF_index,
    100 * eigenvalue_spectrum[:num_modes],
    100 * spectrum_error[:num_modes],
    fmt="o",
    lw=2.5,
    color="black",
)

ax.set_ylabel("Variance Explained (%)")
ax.set_xlabel("Mode")

# Configure and label axes
for axis in ["top", "bottom", "left", "right"]:
    ax.spines[axis].set_linewidth(4)

ax.tick_params(
    axis="both",
    which="major",
    length=12,
    width=4,
    color="#bcbcbc",
    direction="in",
    right=True,
    top=True,
    pad=5,
)

# eof_text =  (f'EOF 1: {explained_variance[0]:0.0f}% of variance \n'
#             +f'EOF 2: {explained_variance[1]:0.0f}% of variance ')
# # Add text showing percentage of explained variance
# ax.text(9.95,0.95, eof_text, fontsize=24,
#        bbox=dict(boxstyle='round', facecolor='#eeeeee', edgecolor='#bcbcbc', alpha=0.5),
#        horizontalalignment='right', verticalalignment='top')

ax.set_ylim(0, 15)
ax.set_xlim(0, 11)
ax.set_xticks(EOF_index)
ax.set_yticks(np.arange(0, 17, 2))

plt.tight_layout()

if not savefig:
    plt.show()
else:
    save_string = (
        f"{experiment}"
      + f"_{compositing_variable.attrs['file_id']}"
      + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}_eigenvalue-spectra.png"
        )
    print(f"Saving plot as {save_string}")
    plt.savefig(
        f"{output_directory}/EOFs/single-variable-latitude-longitude/{save_string}",
        dpi=500,
        bbox_inches="tight",
    )

print(f"{'='*40}")
print("Finished")

### PC Power Spectra

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = True

print(f"Variable: {compositing_variable.name}")

SEGMENT_LENGTH = 256
OVERLAP = SEGMENT_LENGTH // 2
frequency = {}
spectrum = {}
periods = np.array([100, 60, 30, 20, 15, 10, 5, 3, 2])
frequency_from_period = 1 / periods

for i in range(1, 5):
    (frequency[i], spectrum[i]) = signal.welch(
        (PC[i - 1] - np.mean(PC[i - 1])),
        fs=1,
        detrend='linear',
        window="hann",
        nperseg=SEGMENT_LENGTH,
        noverlap=OVERLAP,
    )

# Plot the power spectra
plt.style.use("bmh")

[fig, ax] = plt.subplots(figsize=(16, 9))
ax.set_title(f"Power Spectra of {compositing_variable.attrs['file_id']} Principal Components 1-4", pad=15)
for i in spectrum:
    ax.plot(frequency[i], frequency[i] * spectrum[i], label=("PC" + str(i)), lw=4)

ax.tick_params(
    which="major", direction="in", color="#bcbcbc", length=8, width=2, pad=15
)

ax.tick_params(
    which="minor", direction="in", color="#bcbcbc", length=4, width=1, pad=15
)

# Add vertical lines to show the intraseasonal band
for period in [100, 20]:
    ax.axvline(x=1 / period, ls=":", lw=4, color="black", alpha=0.5)

# Configure axes
ax.set_xscale("log")
ax.set_ylabel(rf"[{compositing_variable.attrs['units']}]$^{2}$")
ax.set_xlabel("Period (days)")
ax.legend(loc="best")

for edge in ["top", "bottom", "left", "right"]:
    ax.spines[edge].set_linewidth(4)

ax.tick_params(
    which="major",
    axis="both",
    direction="in",
    top=True,
    right=True,
    width=4,
    length=15,
    color="#bcbcbc",
    pad=10,
)

# Set tick parameters
ax.set_xticks(ticks=frequency_from_period)
ax.xaxis.set_major_formatter(mticker.FixedFormatter(periods))
ax.set_xlim(1 / 150, 1 / 5)

plt.tight_layout()

if not savefig:
    plt.show()
else:
    save_string = (
        f"{experiment}"
      + f"_{compositing_variable.attrs['file_id']}"
      + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}_power-spectra.png"
        )
    print(f"Saving plot as {save_string}")
    plt.savefig(
        f"{output_directory}/EOFs/single-variable-latitude-longitude/{save_string}",
        dpi=500,
        bbox_inches="tight",
    )

print(f"{'='*40}")
print("Finished")

## Fields regressed onto PC time series

In [ ]:
variables_regressed = {}

print(f"{'Regressions':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")
for regression_PC in [1,2]:
    print(f'{f"Regressing onto {compositing_variable.name} PC{regression_PC}":^{config.SEP_WIDTH}}')
    print(f"{'='*config.SEP_WIDTH}")
    variables_regressed[regression_PC] = {}
    for index, variable in enumerate(variables_filtered):
        print(f'{f"({index+1}/{len(variables_filtered)}) {variable}...":<{config.SEP_WIDTH-1}}', end="")
        variables_regressed[regression_PC][variable] = variables_filtered[variable].copy(deep=True).isel(time=0, drop=True)

        variables_regressed[regression_PC][variable].values = np.einsum(
            'i..., i -> ...',
            (variables_filtered[variable] - variables_filtered[variable].mean(dim='time')),
            standardize_time_series(PC[regression_PC-1])
        ) / len(time)
        print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

In [ ]:
for variable in variables_to_plot:
    print({k: copy.deepcopy(v) for k, v in {
                    "cmap": plotting_attributes[variable.name].get("d_cmap")
                }.items() if v is not None})

#### Latitude-Longitude

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

print(f"Experiment SST: {experiment}")
print(f"{'='*config.SEP_WIDTH}")

for regression_PC in [1,2]:
    print(f"Regressions onto PC{regression_PC}")

    variables_to_plot = [
        variables_regressed[regression_PC]['precipitation'.title()],
        variables_regressed[regression_PC]['outgoing longwave radiation'.title()],
        variables_regressed[regression_PC]['zonal wind'.title()].sel(plev=200),
        variables_regressed[regression_PC]['zonal wind'.title()].sel(plev=850),
        variables_regressed[regression_PC]['vertical wind'.title()].sel(plev=500),
        variables_regressed[regression_PC]['column temperature'.title()],
        variables_regressed[regression_PC]['column water vapor'.title()],
        variables_regressed[regression_PC]['moist static energy'.title()].sel(plev=850),
        variables_regressed[regression_PC]['column longwave heating'.title()],
        variables_regressed[regression_PC]['column shortwave heating'.title()],
        variables_regressed[regression_PC]['latent heat flux'.title()],
        variables_regressed[regression_PC]['sensible heat flux'.title()],
    ]

    for variable in variables_to_plot:
        print(f"Variable: {variable.name}")
        plt.style.use("default")
        plt.rcParams.update({"font.size": 24})

        fig = plt.figure(figsize=(16, 5))
        gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
        gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.0)

        ax = fig.add_subplot(gs[0])
        cb_ax = fig.add_subplot(gs[1])

        ax.set_title(
                f"{experiment}"
              + f"{(' ' + (str(variable.plev.values) + '-hPa') if 'plev' in variable.coords else '')}"
              + f" {variable.name} Regressed onto {compositing_variable.name} PC{regression_PC}",
                pad=15
            )

        # Add cyclic point
        cdata, clon = cutil.add_cyclic_point(
            variable,
            coord=variable.lon,
        )

        # Plot data
        im = ax.contourf(
            clon,
            variable.lat,
            cdata,
            levels=16,
            norm=mcolors.CenteredNorm(vcenter=0),
            **{k: copy.deepcopy(v) for k, v in {
                "cmap": plotting_attributes[variable.name].get("d_cmap")
            }.items() if v is not None}
        )

        # Add colorbar
        cbar = fig.colorbar(
            im,
            cax=cb_ax,
            label=variable.attrs['units'],
            orientation="horizontal",
        )
        cbar.ax.tick_params(labelsize=20)
        # cbar.set_ticks(np.arange(1, 16, 2))

        # Axis parameters
        ax.set_aspect("equal")

        ax.set_xlim(0, 360)
        x_ticks = np.arange(0, 360 + 60, 60)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.set_xlabel("Longitude")

        ax.set_ylim(-30, 30)
        y_ticks = np.arange(-30, 45, 15)
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(tick_labeller(y_ticks, "lat"))
        ax.set_ylabel("Latitude")

        grid_kwargs = {"linewidth": 1, "linestyle": (0, (5, 10)), "color": "gray"}
        ax.grid(True, **grid_kwargs)

        if not savefig:
            plt.show()
        else:
            save_string = (
                f"{experiment}"
                + f"_{variable.attrs['file_id']}"
                + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}"
                + f"_regressed-on-{compositing_variable.attrs['file_id']}-PC{regression_PC}"
                + f".png"
            )
            print(f"Saving plot as {save_string}")
            plt.savefig(
                f"{output_directory}/regressions/latitude-longitude/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )
    print(f"{'='*20}")

print(f"{'='*40}")
print("Finished")

#### Longitude-Height

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False
meridional_mean_region = slice(-10, 10)

print(f"Experiment SST: {experiment}")
print(f"{'='*config.SEP_WIDTH}")

for regression_PC in [1,2]:
    print(f"Regressions onto PC{regression_PC}")

    variables_to_plot = [
        variables_regressed[regression_PC]['zonal wind'.title()],
        variables_regressed[regression_PC]['meridional wind'.title()],
        variables_regressed[regression_PC]['vertical wind'.title()],
        variables_regressed[regression_PC]['temperature'.title()],
        variables_regressed[regression_PC]['moisture'.title()],
        variables_regressed[regression_PC]['geopotential height'.title()],
        variables_regressed[regression_PC]['moist static energy'.title()],
        variables_regressed[regression_PC]['longwave heating rate'.title()],
        variables_regressed[regression_PC]['shortwave heating rate'.title()],
    ]

    for index, variable in enumerate(variables_to_plot):
        print(f"({index+1}/{len(variables_to_plot)}) {variable.name}")
        plt.style.use("default")
        plt.rcParams.update({"font.size": 24})

        fig = plt.figure(figsize=(16, 9))
        gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
        gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.32, wspace=0.0)

        ax = fig.add_subplot(gs[0])
        cb_ax = fig.add_subplot(gs[1])

        ax.set_title(
            (
                f"{experiment} Meridional-Mean"
              + f" ({tick_labeller([meridional_mean_region.start], 'lat')[0]}"
              + f"-{tick_labeller([meridional_mean_region.stop], 'lat')[0]})"
              + f" {variable.name.title()} \nRegressed onto {compositing_variable.name} PC{regression_PC}"
            ),
            pad=15
        )

        # Add cyclic point
        cdata, clon = cutil.add_cyclic_point(
            variable.sel(lat=meridional_mean_region).mean(dim=['lat']).T,
            coord=variable.lon,
        )

        # Plot data
        im = ax.contourf(
            clon,
            variable.plev,
            cdata,
            levels=16,
            norm=mcolors.CenteredNorm(vcenter=0),
            **{k: copy.deepcopy(v) for k, v in {
                "cmap": plotting_attributes[variable.name].get("d_cmap")
            }.items() if v is not None}
        )

        # Add colorbar
        cbar = fig.colorbar(
            im,
            cax=cb_ax,
            label=variable.attrs['units'],
            orientation="horizontal",
        )
        cbar.ax.tick_params(labelsize=20)

        # Axis parameters
        ax.set_xlim(0, 360)
        x_ticks = np.arange(0, 360 + 60, 60)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(tick_labeller(x_ticks-180, "lon"))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.set_xlabel("Longitude")

        ax.set_ylim(100, 950)
        ax.set_ylabel("Pressure (hPa)")
        ax.invert_yaxis()

        if not savefig:
            plt.show()
        else:
            save_string = (
                f"{experiment}_time-mean_meridional-mean-{tick_labeller([meridional_mean_region.start], 'lat', False)[0]}"
              + f"-{tick_labeller([meridional_mean_region.stop], 'lat', False)[0]}"
              + f"_{variable.attrs['file_id']}"
              + f"_regressed-on-{compositing_variable.attrs['file_id']}-PC{regression_PC}"
              + f".png"
                )
            print(f"Saving plot as {save_string}")
            plt.savefig(
                f"{output_directory}/regressions/longitude-height/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )

    print(f"{'='*config.SEP_WIDTH}")
print(f"{'='*config.SEP_WIDTH}")
print("Finished")

#### Latitude-Height

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

print(f"Experiment SST: {experiment}")
print(f"{'='*config.SEP_WIDTH}")

for regression_PC in [1,2]:
    print(f"Regressions onto PC{regression_PC}")

    variables_to_plot = [
        variables_regressed[regression_PC]['zonal wind'.title()],
        variables_regressed[regression_PC]['meridional wind'.title()],
        variables_regressed[regression_PC]['vertical wind'.title()],
        variables_regressed[regression_PC]['temperature'.title()],
        variables_regressed[regression_PC]['moisture'.title()],
        variables_regressed[regression_PC]['geopotential height'.title()],
        variables_regressed[regression_PC]['moist static energy'.title()],
        variables_regressed[regression_PC]['longwave heating rate'.title()],
        variables_regressed[regression_PC]['shortwave heating rate'.title()],
    ]

    for index, variable in enumerate(variables_to_plot):
        print(f"({index+1}/{len(variables_to_plot)}) {variable.name}")
        plt.style.use("default")
        plt.rcParams.update({"font.size": 24})

        fig = plt.figure(figsize=(16, 9))
        gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
        gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.32, wspace=0.0)

        ax = fig.add_subplot(gs[0])
        cb_ax = fig.add_subplot(gs[1])

        ax.set_title(
            f"{experiment} Time-Mean Zonal-Mean {variable.name.title()}"
          + f" \nRegressed on {compositing_variable.name} PC{regression_PC}",
            pad=15
        )

        # Plot data
        im = ax.contourf(
            variable.lat,
            variable.plev,
            variable.mean(dim=['lon']).T,
            levels=16,
            norm=mcolors.CenteredNorm(vcenter=0),
            **{k: copy.deepcopy(v) for k, v in {
                "cmap": plotting_attributes[variable.name].get("d_cmap")
            }.items() if v is not None}
        )

        # Add colorbar
        cbar = fig.colorbar(
            im,
            cax=cb_ax,
            label=variable.attrs['units'],
            orientation="horizontal",
        )
        cbar.ax.tick_params(labelsize=20)

        # Axis parameters
        ax.set_xlim(-30, 30)
        x_ticks = np.arange(-30, 30+10, 10)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(tick_labeller(x_ticks, "lat"))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
        ax.set_xlabel("Latitude")

        # ax.set_yscale('log')
        ax.set_ylim(100, 950)
        ax.set_ylabel("Pressure (hPa)")
        ax.invert_yaxis()

        if not savefig:
            plt.show()
        else:
            save_string = (
                f"{experiment}_time-mean_zonal-mean"
              + f"_{variable.attrs['file_id']}"
              + f"_regressed-on-{compositing_variable.attrs['file_id']}-PC{regression_PC}"
              + f".png"
                )
            print(f"Saving plot as {save_string}")
            plt.savefig(
                f"{output_directory}/regressions/latitude-height/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )

    print(f"{'='*config.SEP_WIDTH}")
print(f"{'='*config.SEP_WIDTH}")
print("Finished")

## Combined meridional-mean EOFs

### Calculate EOFs

In [ ]:
near_equatorial_variables = {}

print(f"{'Combined EOF Analysis':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

print("Meridionally averaging...")
for variable in variables_filtered:
    near_equatorial_variables[variable] = (
        variables_filtered[variable].sel(lat=slice(-15, 15)).mean(dim="lat")
    )
    near_equatorial_variables[variable] /= np.sqrt(
        near_equatorial_variables["precipitation"].var(dim="time").mean(dim="lon")
    )

print("Combining data...")
combined_data = np.concatenate(
    [
        near_equatorial_variables["outgoing longwave radiation"],
        near_equatorial_variables["zonal wind"].sel(plev=200),
        near_equatorial_variables["zonal wind"].sel(plev=850),
    ],
    axis=1,
)

#### Calculate EOFs and PCs
print("Computing EOFs...")
U_combined, S_combined, VT_combined = np.linalg.svd(combined_data.T, full_matrices=False)
EOF_combined = U_combined.T
PC_combined = np.dot(np.diag(S_combined), VT_combined)

# Calculate nominal degrees of freedom
nominal_degrees_of_freedom_combined = np.size(combined_data, 1)

# Calculate eigenvalues and spectrum
eigenvalues_combined = S_combined**2 / nominal_degrees_of_freedom_combined
eigenvalue_spectrum_combined = eigenvalues_combined / np.sum(eigenvalues_combined)
explained_variance_combined = 100 * eigenvalue_spectrum_combined

# Estimate 1-lag autocorrelation and effective degrees of freedom
lag = 1
B = 0
for k in range(lag - 1, nominal_degrees_of_freedom_combined - lag):
    B = B + np.sum(combined_data[:, k] * combined_data[:, k + lag])
phi_L_combined = 1 / (nominal_degrees_of_freedom_combined - 2 * lag) * B
phi_0_combined = 1 / nominal_degrees_of_freedom_combined * np.sum(combined_data**2)
autocorrelation_combined = phi_L_combined / phi_0_combined
degrees_of_freedom_combined = (
    (1 - autocorrelation_combined**2) / (1 + autocorrelation_combined**2)
) * nominal_degrees_of_freedom_combined

# Estimate uncertainty in eigenvalue spectrum
spectrum_error_combined = eigenvalue_spectrum_combined * np.sqrt(2 / degrees_of_freedom_combined)

# Extract the EOFs of each variable from the array
olr_EOF = EOF_combined[:, : len(longitude)]
upper_level_zonal_wind_EOF = EOF_combined[:, len(longitude) : 2 * len(longitude)]
lower_level_zonal_wind_EOF = EOF_combined[:, 2 * len(longitude) :]

print("EOFs computed")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

### Plot EOF maps

In [ ]:
savefig=False

plt.style.use('bmh')
plt.rcParams.update({"font.size": 16})

n_eofs_to_plot = 4

fig = plt.figure(figsize=(16, 6*n_eofs_to_plot))
gs = GridSpec(n_eofs_to_plot, 1, figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.30)

fig.suptitle("EOFs of Intraseasonally filtered Tropical data", fontsize=24, y=1.005)

ax = []
for row in range(n_eofs_to_plot):
    ax.append(fig.add_subplot(gs[row, 0]))

for row in range(n_eofs_to_plot):

    ax[row].set_title(f"EOF {row+1}, Explained Variance = {explained_variance_combined[row]:0.1f}%", pad=15)
    ax[row].plot(
        longitude,
        -olr_EOF[row],
        lw=4,
        label="OLR"
    )
    ax[row].plot(
        longitude,
        lower_level_zonal_wind_EOF[row],
        ls="--",
        lw=4,
        label="U850"
    )
    ax[row].plot(
        longitude,
        upper_level_zonal_wind_EOF[row],
        ls="--",
        lw=4,
        label="U200"
    )
    ax[row].axhline(y=0, lw=2, color="k")

# Configure axes
for axis in range(n_eofs_to_plot):
    # Set axis labels and limits
    ax[axis].set_xlabel("Longitude")
    ax[axis].set_ylabel("Normalized Magnitude")
    ax[axis].set_xlim(0, 360-2.5)
    ax[axis].set_ylim(-0.2, 0.2)
    ax[axis].set_xticks(
        ticks=np.arange(0, 360+60, 60),
        labels=tick_labeller(np.arange(0, 360+60, 60), 'lon')
    )

    # Specify tick parameters
    ax[axis].tick_params(
        which="major",
        width=3,
        length=15,
        direction="in",
        color='#bcbcbc',
        top=True,
        right=True,
        pad=10
    )

    ax[axis].tick_params(
        which="minor",
        width=3,
        length=7.5,
        direction="in",
        color='#bcbcbc',
        top=True,
        right=True,
        pad=10
    )

    for edge in ['top','bottom','left','right']:
        ax[axis].spines[edge].set_linewidth(4)

    ax[axis].legend(loc="upper right")

if not savefig:
    plt.show()
else:
    save_string = (
        f"{experiment}_combined-variables_EOF-maps.png"
      # + f"_{variable.attrs['file_id']}"
      # + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}_EOF-maps.png"
        )
    print(f"Saving plot as {save_string}")
    plt.savefig(
        f"{output_directory}/EOFs/combined-variables/{save_string}",
        dpi=500,
        bbox_inches="tight",
    )

print(f"{'='*40}")
print("Finished")

### Plot eigenvalue spectra

In [ ]:
savefig = False

plt.style.use("bmh")
plt.rcParams.update({"font.size": 24})

[fig, ax] = plt.subplots(figsize=(16, 9))
num_modes = 10
EOF_index = np.arange(1, num_modes + 1, 1)
# ax.set_title(
#     (
#         f"Percentage variance, {data_source[plotting_variable]} {variable_file_id[plotting_variable]}, {season_select.capitalize()}\n"
#         + f"{degrees_of_freedom:0.0f} effective degrees of freedom"
#     ),
#     pad=15
# )

ax.bar(EOF_index, 100 * eigenvalue_spectrum_combined[:num_modes], width=0.5, color="dodgerblue")

ax.errorbar(
    EOF_index,
    100 * eigenvalue_spectrum_combined[:num_modes],
    100 * spectrum_error_combined[:num_modes],
    fmt="o",
    lw=2.5,
    color="black",
)

ax.set_ylabel("Variance Explained (%)")
ax.set_xlabel("Mode")

# Configure and label axes
for axis in ["top", "bottom", "left", "right"]:
    ax.spines[axis].set_linewidth(4)

ax.tick_params(
    axis="both",
    which="major",
    length=12,
    width=4,
    color="#bcbcbc",
    direction="in",
    right=True,
    top=True,
    pad=5,
)

# eof_text = (
#     f'EOF 1: {explained_variance_combined[0]:0.0f}% of variance \n'
#     + f'EOF 2: {explained_variance_combined[1]:0.0f}% of variance '
# )

# # Add text showing percentage of explained variance
# ax.text(
#     9.95, 0.05, eof_text, fontsize=24,
#     bbox=dict(
#         boxstyle='round', facecolor='#eeeeee', edgecolor='#bcbcbc', alpha=0.5
#     ),
#     horizontalalignment='right', verticalalignment='top'
# )

ax.set_ylim(0, 25)
ax.set_xlim(0, 11)
ax.set_xticks(EOF_index)
ax.set_yticks(np.arange(0, 25, 2))

plt.tight_layout()

# if not savefig:
#     plt.show()
# else:
#     plt.savefig(
#         (
#             f"{output_directory}/"
#             + f"{season_select}_{data_source[plotting_variable]}-{variable_file_id[plotting_variable]}_percentage-variance.png"
#         ),
#         dpi=300,
#         bbox_inches="tight",
#     )

if not savefig:
    plt.show()
else:
    save_string = (
        f"{experiment}_combined-variables_eigenvalue-spectra.png"
      # + f"_{variable.attrs['file_id']}"
      # + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}_EOF-maps.png"
        )
    print(f"Saving plot as {save_string}")
    plt.savefig(
        f"{output_directory}/EOFs/combined-variables/{save_string}",
        dpi=500,
        bbox_inches="tight",
    )

print(f"{'='*40}")
print("Finished")

In [ ]:
SEGMENT_LENGTH = 256
OVERLAP = SEGMENT_LENGTH // 2
frequency_combined = {}
spectrum_combined = {}
periods = np.array([100, 60, 30, 20, 15, 10, 5, 3, 2])
frequency_from_period = 1 / periods

for i in range(1, 5):
    (frequency_combined[i], spectrum_combined[i]) = signal.welch(
        PC_combined[i - 1] - np.mean(PC_combined[i - 1]),
        fs=1,
        window="hann",
        nperseg=SEGMENT_LENGTH,
        noverlap=OVERLAP,
    )

# Plot the power spectra
plt.style.use("bmh")

[fig, ax] = plt.subplots(figsize=(16, 9))
ax.set_title("Power Spectra of Principal Components", pad=10)
for i in spectrum_combined:
    ax.plot(
        frequency_combined[i],
        frequency_combined[i] * spectrum_combined[i],
        label=f"PC {i}",
        lw=4
    )

ax.tick_params(
    which="major", direction="in", color="#bcbcbc", length=8, width=2, pad=15
)

ax.tick_params(
    which="minor", direction="in", color="#bcbcbc", length=4, width=1, pad=15
)

# Add vertical lines to show the intraseasonal band
for period in [100, 20]:
    ax.axvline(x=1 / period, ls=":", lw=4, color="black", alpha=0.5)

# Configure axes
ax.set_xscale("log")
ax.set_ylabel("Power")
ax.set_xlabel("Period (days)")
ax.legend(loc="best")

for edge in ["top", "bottom", "left", "right"]:
    ax.spines[edge].set_linewidth(4)

ax.tick_params(
    which="major",
    axis="both",
    direction="in",
    top=True,
    right=True,
    width=4,
    length=15,
    color="#bcbcbc",
    pad=10,
)

# Set tick parameters
ax.set_xticks(ticks=frequency_from_period)
ax.xaxis.set_major_formatter(mticker.FixedFormatter(periods))
ax.set_xlim(1/120, 1/5)

plt.show()

# plt.savefig(
#     f"{output_directory}/eof-analysis_principal-component_power-spectra.png",
#     dpi=300,
#     bbox_inches='tight'
# )

## RMM Indices

### Compute Indices

In [ ]:
EOFs_to_use = 'horizontal'

print(f"Constructing RMM indices from {EOFs_to_use} PCs...")
if EOFs_to_use == 'horizontal':
    RMM1 = PC[0] / np.std(PC[0])
    RMM2 = -PC[1] / np.std(PC[1])
    RMM3 = PC[2] / np.std(PC[2])
    RMM4 = -PC[3] / np.std(PC[3])

elif EOFs_to_use == 'combined':
    RMM1 = PC_combined[0] / np.std(PC_combined[0])
    RMM2 = -PC_combined[1] / np.std(PC_combined[1])
    RMM3 = PC_combined[2] / np.std(PC_combined[2])
    RMM4 = -PC_combined[3] / np.std(PC_combined[3])

mjo_strength_12 = np.sqrt(RMM1**2 + RMM2**2)
mjo_strength_34 = np.sqrt(RMM3**2 + RMM4**2)

# Remove weak MJO events
RMM1[mjo_strength_12 < 1] = np.nan
RMM2[mjo_strength_12 < 1] = np.nan
RMM3[mjo_strength_34 < 1] = np.nan
RMM4[mjo_strength_34 < 1] = np.nan

RMM1 = xr.DataArray(
    data=RMM1,
    coords={"time": time},
    attrs=dict(
        EOF_type=EOFs_to_use,
        EOF_vars=(compositing_variable.name if EOFs_to_use == 'horizontal' else 'OLR, U200, U850')
    )
)
RMM2 = xr.DataArray(
    data=RMM2,
    coords={"time": time},
    attrs=dict(
        EOF_type=EOFs_to_use,
        EOF_vars=(compositing_variable.name if EOFs_to_use == 'horizontal' else 'OLR, U200, U850')
    )
)
RMM3 = xr.DataArray(
    data=RMM3,
    coords={"time": time},
    attrs=dict(
        EOF_type=EOFs_to_use,
        EOF_vars=(compositing_variable.name if EOFs_to_use == 'horizontal' else 'OLR, U200, U850')
    )
)
RMM4 = xr.DataArray(
    data=RMM4,
    coords={"time": time},
    attrs=dict(
        EOF_type=EOFs_to_use,
        EOF_vars=(compositing_variable.name if EOFs_to_use == 'horizontal' else 'OLR, U200, U850')
    )
)
print("RMM indices constructed")

### Plot indices

In [ ]:
# # Configure plot
# [fig, ax] = plt.subplots(figsize=(16, 16))
# plt.rcParams["axes.edgecolor"] = "black"
# plt.rcParams["axes.linewidth"] = 3
# ax.set_xlim(-4, 4)
# ax.set_ylim(-4, 4)
# plt.xlabel("RMM1")
# plt.ylabel("RMM2")
# ax.set_facecolor("white")

# # Plot index points
# colormap = plt.get_cmap("viridis")  # sns.color_palette("viridis", as_cmap=True)
# start_index = 0
# end_index = len(RMM1) - 1
# # end_index = 300
# ax.plot(RMM1[start_index], RMM2[start_index], color="black", marker=".", ls="-", ms=30)
# for i in range(start_index + 1, end_index + 1):
#     ax.plot(
#         RMM1[i],
#         RMM2[i],
#         color=colormap((i - start_index) / (end_index - start_index)),
#         marker="o",
#         ms=10,
#     )

# # Add phase regions overlay
# circle1 = plt.Circle((0, 0), 1.0, color="k", fill=False, lw=3, zorder=10)
# ax.hlines(y=0, xmin=-4, xmax=-1, color="k", lw=3, ls="-")
# ax.hlines(y=0, xmin=1, xmax=4, color="k", lw=3, ls="-")
# ax.vlines(x=0, ymin=-4, ymax=-1, color="k", lw=3, ls="-")
# ax.vlines(x=0, ymin=1, ymax=4, color="k", lw=3, ls="-")
# ax.plot([np.sqrt(2) / 2, 4], [np.sqrt(2) / 2, 4], color="k", lw=3, ls="-")


# x_vals = {1: -1, 2: -1, 3: 1, 4: 1, 5: 1, 6: 1, 7: -1, 8: -1}

# y_val1 = {
#     1: 0,
#     2: -4,
#     3: -4,
#     4: 0,
#     5: 0,
#     6: np.linspace(0, 4, len(RMM1)),
#     7: np.linspace(0, 4, len(RMM1)),
#     8: 0,
# }

# y_val2 = {
#     1: -1 * np.linspace(0, 4, len(RMM1)),
#     2: -1 * np.linspace(0, 4, len(RMM1)),
#     3: -1 * np.linspace(0, 4, len(RMM1)),
#     4: -1 * np.linspace(0, 4, len(RMM1)),
#     5: 1 * np.linspace(0, 4, len(RMM1)),
#     6: 4,
#     7: 4,
#     8: 1 * np.linspace(0, 4, len(RMM1)),
# }

# # Fill one of the phases in with blue
# # val=2
# # ax.fill_between(
# #     x_vals[val]*np.linspace(0, 4, len(RMM1)),
# #     y_val1[val],
# #     y_val2[val]
# # )

# # x = np.linspace(-1,1,100)
# # ax.fill_between(x, -np.sqrt(1-x**2), np.sqrt(1-x**2), color='white')

# # Add lines to differentiate the phases
# ax.plot([np.sqrt(2) / 2, 4], [-np.sqrt(2) / 2, -4], color="k", lw=3, ls="-")
# ax.plot([-4, -np.sqrt(2) / 2], [4, np.sqrt(2) / 2], color="k", lw=3, ls="-")
# ax.plot([-4, -np.sqrt(2) / 2], [-4, -np.sqrt(2) / 2], color="k", lw=3, ls="-")
# ax.add_patch(circle1)

# # Add phase labels
# ax.text(
#     -3.5, -0.26, f"Phase 1", horizontalalignment="center", verticalalignment="center"
# )
# ax.text(
#     -0.51, -3.75, f"Phase 2", horizontalalignment="center", verticalalignment="center"
# )
# ax.text(
#     0.5, -3.75, f"Phase 3", horizontalalignment="center", verticalalignment="center"
# )
# ax.text(
#     3.5, -0.26, f"Phase 4", horizontalalignment="center", verticalalignment="center"
# )
# ax.text(3.5, 0.25, f"Phase 5", horizontalalignment="center", verticalalignment="center")
# ax.text(0.5, 3.75, f"Phase 6", horizontalalignment="center", verticalalignment="center")
# ax.text(
#     -0.51, 3.75, f"Phase 7", horizontalalignment="center", verticalalignment="center"
# )
# ax.text(
#     -3.5, 0.25, f"Phase 8", horizontalalignment="center", verticalalignment="center"
# )

# # Add MJO-location labels
# # ax.text(0, 3.5, f'Western Pacific',  horizontalalignment='center',
# #      verticalalignment='center', bbox=props, fontsize=14)

# # ax.text(-3.3, 0, f'Western Hemisphere \n & Africa',  horizontalalignment='center',
# #      verticalalignment='center', bbox=props, fontsize=14)

# # ax.text(3.3, 0, f'Maritime Continent',  horizontalalignment='center',
# #      verticalalignment='center', bbox=props, fontsize=14)

# # ax.text(0, -3.5, f'Indian Ocean',  horizontalalignment='center',
# #      verticalalignment='center', bbox=props, fontsize=14)

# ax.set_aspect("equal")
# plt.tight_layout()

# plt.show()

# # plt.savefig(
# #     f"{output_directory}/eof-analysis_rmm_plot.png",
# #     dpi=300,
# #     bbox_inches='tight'
# # )

In [ ]:
#### Calculate phases
def compute_mjo_phase_indices(RMM1, RMM2):
    phase_times = {}

    # Find times by phase
    phase_times[1] = time.where(
        (RMM1 < 0) & (RMM2 < 0) & (np.abs(RMM1) > np.abs(RMM2))
    ).dropna(dim="time")
    phase_times[2] = time.where(
        (RMM1 < 0) & (RMM2 < 0) & (np.abs(RMM1) < np.abs(RMM2))
    ).dropna(dim="time")
    phase_times[3] = time.where(
        (RMM1 > 0) & (RMM2 < 0) & (np.abs(RMM1) < np.abs(RMM2))
    ).dropna(dim="time")
    phase_times[4] = time.where(
        (RMM1 > 0) & (RMM2 < 0) & (np.abs(RMM1) > np.abs(RMM2))
    ).dropna(dim="time")
    phase_times[5] = time.where(
        (RMM1 > 0) & (RMM2 > 0) & (np.abs(RMM1) > np.abs(RMM2))
    ).dropna(dim="time")
    phase_times[6] = time.where(
        (RMM1 > 0) & (RMM2 > 0) & (np.abs(RMM1) < np.abs(RMM2))
    ).dropna(dim="time")
    phase_times[7] = time.where(
        (RMM1 < 0) & (RMM2 > 0) & (np.abs(RMM1) < np.abs(RMM2))
    ).dropna(dim="time")
    phase_times[8] = time.where(
        (RMM1 < 0) & (RMM2 > 0) & (np.abs(RMM1) > np.abs(RMM2))
    ).dropna(dim="time")

    # Find indices by phase
    phase_indices = {}

    # Find all of the points in phase 1
    phase_indices[1] = np.squeeze(
        np.where((RMM1 < 0) & (RMM2 < 0) & (np.abs(RMM1) > np.abs(RMM2)))
    )

    # Phase 2
    phase_indices[2] = np.squeeze(
        np.where((RMM1 < 0) & (RMM2 < 0) & (np.abs(RMM1) < np.abs(RMM2)))
    )

    # Phase 3
    phase_indices[3] = np.squeeze(
        np.where((RMM1 > 0) & (RMM2 < 0) & (np.abs(RMM1) < np.abs(RMM2)))
    )

    # Phase 4
    phase_indices[4] = np.squeeze(
        np.where((RMM1 > 0) & (RMM2 < 0) & (np.abs(RMM1) > np.abs(RMM2)))
    )

    # Phase 5
    phase_indices[5] = np.squeeze(
        np.where((RMM1 > 0) & (RMM2 > 0) & (np.abs(RMM1) > np.abs(RMM2)))
    )

    # Phase 6
    phase_indices[6] = np.squeeze(
        np.where((RMM1 > 0) & (RMM2 > 0) & (np.abs(RMM1) < np.abs(RMM2)))
    )

    # Phase 7
    phase_indices[7] = np.squeeze(
        np.where((RMM1 < 0) & (RMM2 > 0) & (np.abs(RMM1) < np.abs(RMM2)))
    )

    # Phase 8
    phase_indices[8] = np.squeeze(
        np.where((RMM1 < 0) & (RMM2 > 0) & (np.abs(RMM1) > np.abs(RMM2)))
    )

    return phase_indices, phase_times

## MJO Composites

#### Longitude-Latitude

In [ ]:
savefig = False
RMM_pair = '1-2'
[phase_indices, phase_times] = compute_mjo_phase_indices(
    (RMM1 if RMM_pair == '1-2' else RMM3),
    (RMM2 if RMM_pair == '1-2' else RMM4)
)

variables_to_plot=[
    variables_subset['precipitation'.title()],
    variables_subset['outgoing longwave radiation'.title()],
    variables_subset['zonal wind'.title()].sel(plev=200),
    variables_subset['zonal wind'.title()].sel(plev=850),
    variables_subset['meridional wind'.title()].sel(plev=850),
    variables_subset['vertical wind'.title()].sel(plev=500),
    variables_subset['column temperature'.title()],
    variables_subset['column water vapor'.title()],
    variables_subset['geopotential height'.title()].sel(plev=500),
    variables_subset['moist static energy'.title()].sel(plev=850),
    variables_subset['column longwave heating'.title()],
    variables_subset['column shortwave heating'.title()],
    variables_subset['latent heat flux'.title()],
    variables_subset['sensible heat flux'.title()],

]


for variable in variables_to_plot:
    print(f"Variable: {variable.name}")
    variable_anomalies = variable - variable.mean(dim=["lat", "lon"])

    variables_by_phase = {}

    for phase in range(1, 9):
        variables_by_phase[phase] = variable_anomalies.sel(time=phase_times[phase]).mean(dim="time")

    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})
    plt.rcParams["figure.dpi"] = 300
    coastline_width = 1
    props = dict(boxstyle="round", facecolor="#eeeeee")

    data_crs = ccrs.PlateCarree()
    proj = ccrs.PlateCarree(central_longitude=-205)

    gs = GridSpec(8, 2, width_ratios=[100, 3])
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.15)
    fig = plt.figure(figsize=(16, 30))

    fig.suptitle(
        (
            f"{experiment}"
          + f"{(' ' + (str(variable.plev.values) + '-hPa') if 'plev' in variable.coords else '')}"
          + f" {variable.name} composited over\n {RMM1.attrs['EOF_vars']} RMMs {RMM_pair}-derived MJO-phases"
        ),
        x=0.485,
        y=0.9975
    )

    for phase in range(1, 9):
        # Add an axis object for each phase
        ax = fig.add_subplot(gs[phase - 1, 0])

        ax.set_title(f"Phase {phase:0.0f}")

        # Add the cyclic point
        cdata, clon = cutil.add_cyclic_point(
            variables_by_phase[phase],
            coord=variables_by_phase[phase].lon
        )

        # Plot rainfall by phase
        im = ax.contourf(
            clon,
            variables_by_phase[phase].lat,
            cdata,
            levels=16,
            norm=mcolors.CenteredNorm(vcenter=0),
            **{k: copy.deepcopy(v) for k, v in {
                "cmap": plotting_attributes[variable.name].get("d_cmap")
            }.items() if v is not None}
        )

        # Map parameters
        ax.set_xlabel("")

        ax.set_xlim(0, 360)
        ax.set_ylim(-30, 30)

        xtick_locations = np.arange(0, 360 + 60, 60)
        ytick_locations = np.arange(-30, 40, 10)

        ax.set_xticks(
            ticks=xtick_locations,
            # labels=(tick_labeller(xtick_locations - 180, "lon") if phase == 8 else '')
            labels=tick_labeller(xtick_locations - 180, "lon")
        )
        ax.set_yticks(ticks=ytick_locations, labels=tick_labeller(ytick_locations, "lat"))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=7, prune="lower"))

        ax.set_aspect("auto")
        ax.grid(True, **grid_kwargs)

    # Set colorbar
    cbar_ax = fig.add_subplot(gs[1:-1, 1])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.ax.tick_params(labelsize=20)
    # cbar.set_ticks(np.arange(-4, 24, 4))
    cbar.set_label(variable.attrs['units'])

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}"
          + f"_{variable.attrs['file_id']}"
          + f"{((str(variable.plev.values)) if 'plev' in variable.coords else '')}"
          + f"_composited-on-{compositing_variable.attrs['file_id']}"
          + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
          + f"-RMMs-{RMM_pair}.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/composites/latitude-longitude/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

#### Longitude-Height

In [ ]:
savefig = False
RMM_pair = '1-2'
[phase_indices, phase_times] = compute_mjo_phase_indices(
    (RMM1 if RMM_pair == '1-2' else RMM3),
    (RMM2 if RMM_pair == '1-2' else RMM4)
)

meridional_mean_region = slice(-10, 10)

variables_to_plot = [
    variables_subset['zonal wind'.title()],
    variables_subset['meridional wind'.title()],
    variables_subset['vertical wind'.title()],
    variables_subset['temperature'.title()],
    variables_subset['moisture'.title()],
    variables_subset['geopotential height'.title()],
    variables_subset['moist static energy'.title()],
    variables_subset['longwave heating rate'.title()],
    variables_subset['shortwave heating rate'.title()],
]


for variable in variables_to_plot:
    print(f"Variable: {variable.name}")
    meridonal_mean_variables = variable.sel(lat=meridional_mean_region).weighted(weights).mean(dim="lat")
    variable_anomalies = meridonal_mean_variables - meridonal_mean_variables.mean(dim=["lon", "plev"])

    variables_by_phase = {}

    for phase in range(1, 9):
        variables_by_phase[phase] = variable_anomalies.sel(time=phase_times[phase]).mean(dim="time")

    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})
    plt.rcParams["figure.dpi"] = 300
    coastline_width = 1
    props = dict(boxstyle="round", facecolor="#eeeeee")

    data_crs = ccrs.PlateCarree()
    proj = ccrs.PlateCarree(central_longitude=-205)

    gs = GridSpec(8, 2, width_ratios=[100, 3])
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.15)
    fig = plt.figure(figsize=(16, 30))

    fig.suptitle(
        (
            f"{experiment}"
          + f" Meridional Mean"
          + f" ({tick_labeller([meridional_mean_region.start], 'lat')[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat')[0]})"
          + f" {variable.name} composited over\n {RMM1.attrs['EOF_vars']} RMMs {RMM_pair}-derived MJO-phases"
        ),
        x=0.485,
        y=0.9975
    )

    for phase in range(1, 9):
        # Add an axis object for each phase
        ax = fig.add_subplot(gs[phase - 1, 0])

        ax.set_title(f"Phase {phase:0.0f}")

        # Add the cyclic point
        cdata, clon = cutil.add_cyclic_point(
            variables_by_phase[phase].T,
            coord=variables_by_phase[phase].lon
        )

        # Plot rainfall by phase
        im = ax.contourf(
            clon,
            variables_by_phase[phase].plev,
            cdata,
            levels=16,
            norm=mcolors.CenteredNorm(vcenter=0),
            **{k: copy.deepcopy(v) for k, v in {
                "cmap": plotting_attributes[variable.name].get("d_cmap")
            }.items() if v is not None}
        )

        ax.invert_yaxis()
        # Map parameters
        ax.set_xlabel("")
        ax.set_ylabel("hPa")

        ax.set_xlim(0, 360)
        ax.set_ylim(950, 100)

        xtick_locations = np.arange(0, 360 + 60, 60)
        ytick_locations = np.arange(100, 950, 150)

        ax.set_xticks(
            ticks=xtick_locations,
            labels=tick_labeller(xtick_locations - 180, "lon")
        )
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=7, prune="lower"))
        ax.set_yticks(ytick_locations)

        ax.set_aspect("auto")
        ax.grid(True, **grid_kwargs)

    # Set colorbar
    cbar_ax = fig.add_subplot(gs[1:-1, 1])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.ax.tick_params(labelsize=20)
    # cbar.set_ticks(np.arange(-4, 24, 4))
    cbar.set_label(variable.attrs['units'])

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}"
          + f"_meridional-mean-{tick_labeller([meridional_mean_region.start], 'lat', False)[0]}"
          + f"-{tick_labeller([meridional_mean_region.stop], 'lat', False)[0]}"
          + f"_{variable.attrs['file_id']}"
          + f"_composited-on-{compositing_variable.attrs['file_id']}"
          + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
          + f"-RMMs-{RMM_pair}.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/composites/longitude-height/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

#### Latitude-Height

In [ ]:
savefig = False
RMM_pair = '1-2'
[phase_indices, phase_times] = compute_mjo_phase_indices(
    (RMM1 if RMM_pair == '1-2' else RMM3),
    (RMM2 if RMM_pair == '1-2' else RMM4)
)

meridional_mean_region = slice(-10, 10)

variables_to_plot = [
    variables_subset['zonal wind'].title(),
    variables_subset['meridional wind'].title(),
    variables_subset['vertical wind'].title(),
    variables_subset['temperature'].title(),
    variables_subset['moisture'.title()]
]


for variable in variables_to_plot:
    print(f"Variable: {variable.name}")
    zonal_mean_variables = variable.mean(dim="lon")
    variable_anomalies = zonal_mean_variables - zonal_mean_variables.mean(dim=["lat", "plev"])

    variables_by_phase = {}

    for phase in range(1, 9):
        variables_by_phase[phase] = variable_anomalies.sel(time=phase_times[phase]).mean(dim="time")

    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})
    plt.rcParams["figure.dpi"] = 300
    coastline_width = 1
    props = dict(boxstyle="round", facecolor="#eeeeee")

    data_crs = ccrs.PlateCarree()
    proj = ccrs.PlateCarree(central_longitude=-205)

    gs = GridSpec(8, 2, width_ratios=[100, 3])
    gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.15)
    fig = plt.figure(figsize=(16, 30))

    fig.suptitle(
        (
            f"{experiment}"
          + f" Zonal-Mean"
          + f" {variable.name} composited over\n {RMM1.attrs['EOF_vars']} RMMs {RMM_pair}-derived MJO-phases"
        ),
        x=0.485,
        y=0.9975
    )

    for phase in range(1, 9):
        # Add an axis object for each phase
        ax = fig.add_subplot(gs[phase - 1, 0])

        ax.set_title(f"Phase {phase:0.0f}")

        # Add the cyclic point
        cdata, clat = cutil.add_cyclic_point(
            variables_by_phase[phase].T,
            coord=variables_by_phase[phase].lat
        )

        # Plot rainfall by phase
        im = ax.contourf(
            clat,
            variables_by_phase[phase].plev,
            cdata,
            levels=16,
            norm=mcolors.CenteredNorm(vcenter=0),
            **{k: copy.deepcopy(v) for k, v in {
                "cmap": plotting_attributes[variable.name].get("d_cmap")
            }.items() if v is not None}
        )

        ax.invert_yaxis()
        # Map parameters
        ax.set_xlabel("")
        ax.set_ylabel("hPa")

        ax.set_xlim(-30, 30)
        ax.set_ylim(950, 100)

        xtick_locations = np.arange(-30, 30+10, 10)
        ytick_locations = np.arange(100, 950, 150)

        ax.set_xticks(
            ticks=xtick_locations,
            labels=tick_labeller(xtick_locations, "lat")
        )
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=7, prune="lower"))
        ax.set_yticks(ytick_locations)

        ax.set_aspect("auto")
        ax.grid(True, **grid_kwargs)

    # Set colorbar
    cbar_ax = fig.add_subplot(gs[1:-1, 1])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.ax.tick_params(labelsize=20)
    # cbar.set_ticks(np.arange(-4, 24, 4))
    cbar.set_label(variable.attrs['units'])

    if not savefig:
        plt.show()
    else:
        save_string = (
            f"{experiment}"
          + f"_zonal-mean"
          + f"_{variable.attrs['file_id']}"
          + f"_composited-on-{compositing_variable.attrs['file_id']}"
          + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
          + f"-RMMs-{RMM_pair}.png"
            )
        print(f"Saving plot as {save_string}")
        plt.savefig(
            f"{output_directory}/composites/latitude-height/{save_string}",
            dpi=500,
            bbox_inches="tight",
        )

print(f"{'='*40}")
print("Finished")

#### OLR, U850, V850

In [ ]:
[phase_indices, phase_times] = compute_mjo_phase_indices(RMM1, RMM2)

variables_to_plot = [
    variables_subset['outgoing longwave radiation'],
    variables_subset['zonal wind'].sel(plev=850),
    variables_subset['meridional wind'].sel(plev=850),
]

variable_anomalies = {}
variables_by_phase = {}

for variable in variables_to_plot:
    variable_anomalies[variable.name] = variable - variable.mean(dim=["lat", "lon"])

    variables_by_phase[variable.name] = {}
    for phase in range(1, 9):
        variables_by_phase[variable.name][phase] = (
            variable_anomalies[variable.name].sel(time=phase_times[phase]).mean(dim="time")
        )


plt.style.use("default")
plt.rcParams.update({"font.size": 24})
plt.rcParams["figure.dpi"] = 300
coastline_width = 1
props = dict(boxstyle="round", facecolor="#eeeeee")

data_crs = ccrs.PlateCarree()
proj = ccrs.PlateCarree(central_longitude=-205)

gs = GridSpec(8, 2, width_ratios=[100, 3])
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.15)
fig = plt.figure(figsize=(16, 30))

# cmap_modified = modified_colormap('RdYlBu_r', 'white', 0.05, 0.05)
cmap_modified = mjo.modified_colormap("coolwarm", "white", 0.05, 0.05)

keyword_args = {
    # 'transform':data_crs,
    "cmap": cmap_modified,
    # "levels": np.linspace(-55, 25, 21),
    'levels': 21,
    "norm": mcolors.CenteredNorm(vcenter=0),
    # 'extend': 'both'
}

for phase in range(1, 9):
    # Add an axis object for each phase
    # ax = fig.add_subplot(gs[phase-1, 0], projection=proj)
    ax = fig.add_subplot(gs[phase - 1, 0])

    ax.set_title(f"Phase {phase:0.0f}")

    # Add the cyclic point
    cdata, clon = cutil.add_cyclic_point(
        variables_by_phase["Outgoing Longwave Radiation"][phase],
        coord=variables_by_phase["Outgoing Longwave Radiation"][phase].lon,
    )

    # Plot rainfall by phase
    im = ax.contourf(
        clon,
        variables_by_phase["Outgoing Longwave Radiation"][phase].lat,
        cdata,
        **keyword_args,
    )

    cdata_u, clon_u = cutil.add_cyclic_point(
        variables_by_phase["Zonal Wind"][phase],
        coord=variables_by_phase["Zonal Wind"][phase].lon,
    )

    cdata_v, clon_v = cutil.add_cyclic_point(
        variables_by_phase["Meridional Wind"][phase],
        coord=variables_by_phase["Meridional Wind"][phase].lon,
    )

    quiver_lon_skip = 3
    quiver_lat_skip = 4
    q = ax.quiver(
        clon_u[::quiver_lon_skip],
        variables_by_phase["Zonal Wind"][phase].lat[::quiver_lat_skip],
        cdata_u[::quiver_lat_skip, ::quiver_lon_skip],
        cdata_v[::quiver_lat_skip, ::quiver_lon_skip],
    )

    # Map parameters
    ax.set_xlabel("")

    ax.set_xlim(0, 360)
    ax.set_ylim(-30, 30)

    xtick_locations = np.arange(0, 360 + 60, 60)
    ytick_locations = np.arange(-30, 40, 10)

    ax.set_xticks(
        ticks=xtick_locations, labels=tick_labeller(xtick_locations - 180, "lon")
    )
    ax.set_yticks(ticks=ytick_locations, labels=tick_labeller(ytick_locations, "lat"))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=7, prune="lower"))

    ax.set_aspect("auto")
    ax.grid(True, lw=1, ls=(0, (5, 10)), color="gray")

# Set colorbar
cbar_ax = fig.add_subplot(gs[1:-1, 1])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=20)
cbar.set_ticks(im.levels[::2])
cbar.set_label(r"W m$^{-2}$")

plt.show()

# Lag Correlations

In [ ]:
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"
savefig = False

PCs_to_use = [1, 2]

correlation = []
lag_days = np.arange(-30, 31)
for lag in lag_days:
    correlation.append(
        np.correlate(
            standardize_time_series(PC[PCs_to_use[0]-1]),
            np.roll(standardize_time_series(PC[PCs_to_use[1]-1]), shift=lag),
        )
        / len(PC[0])
    )

plt.style.use("bmh")
plt.rcParams.update({"font.size": 20})

[fig, ax] = plt.subplots(figsize=(16, 9))
ax.set_title(
    f"Lag Correlation between {experiment} {compositing_variable.name} PCs {PCs_to_use[0]} & {PCs_to_use[1]}",
    pad=15
)

ax.plot(lag_days, correlation, marker="o")
ax.axhline(y=0, color="#bcbcbc", lw=3, zorder=-10)
ax.set_ylim(-0.8, 0.8)
ax.set_xlim(-30, 30)
ax.set_yticks(np.arange(-0.8, 1, 0.2))
ax.set_xticks(np.arange(-30, 35, 5))
ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=13, prune="lower"))
ax.set_xlabel("Lag (day)")
ax.set_ylabel("Correlation")
for spine in ax.spines.values():
    spine.set_edgecolor("#bcbcbc")
    spine.set_linewidth(3)

if not savefig:
    plt.show()
else:
    save_string = (
        f"{experiment}"
      + f"_{compositing_variable.attrs['file_id']}"
      + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
      + f"_PC-{PCs_to_use[0]}-{PCs_to_use[1]}_lag-correlation.png"
        )
    print(f"Saving plot as {save_string}")
    plt.savefig(
        f"{output_directory}/correlations/PC-lag-correlations/{save_string}",
        dpi=500,
        bbox_inches="tight",
    )

print(f"{'='*40}")
print("Finished")

## Lag-Longitude

In [ ]:
calculate_corr = True
plot_corr = True
savefig = False

meridional_mean_region = slice(-10,10)
weights = np.cos(np.deg2rad(latitude))
weights.name = "weights"
lag_days = np.arange(-30, 31)
reference_lon = 180

variables_to_plot = [
    variables_filtered['precipitation'.title()],
    variables_filtered['outgoing longwave radiation'.title()],
    variables_filtered['zonal wind'.title()].sel(plev=200),
    variables_filtered['zonal wind'.title()].sel(plev=850),
    variables_filtered['vertical wind'.title()].sel(plev=500),
    variables_filtered['column temperature'.title()],
    variables_filtered['column water vapor'.title()],
]

if calculate_corr or 'lag_longitude_correlation' not in globals():
    lag_longitude_correlation = {}

for variable in variables_to_plot:
    print(f"Variable: {variable.name}")

    if calculate_corr:
        lag_longitude_correlation[variable.name] = xr.DataArray(
            data=np.empty((len(lag_days), len(longitude))),
            dims=["lag", "lon"],
            coords=dict(
                lag=lag_days,
                lon=longitude
            )
        )

        # reference_time_series = variable.sel(
        #         lat=meridional_mean_region,
        #         lon=reference_lon
        #     ).weighted(weights).mean(dim="lat")

        reference_time_series = PC[0]

        print("Computing correlations...")
        for lag_index, lag in enumerate(lag_days):
            lag_longitude_correlation[variable.name][lag_index] = np.einsum(
                    'ij,i->j',
                    standardize_time_series(variable.sel(lat=meridional_mean_region).weighted(weights).mean(dim='lat')),
                    np.roll(standardize_time_series(reference_time_series), shift=lag),
                ) / len(reference_time_series)
        print("Correlations computed")

    if plot_corr:
        plt.style.use("bmh")
        plt.rcParams.update({"font.size": 20})
        [fig, ax] = plt.subplots(figsize=(16, 6))

        ax.set_title(
            f"Meridional-Mean {variable.name}"
            + f" ({tick_labeller([meridional_mean_region.start], 'lat')[0]}"
            + f"-{tick_labeller([meridional_mean_region.stop], 'lat')[0]})\n Lag-Longitude Correlation",
            pad=10
        )

        cdata, clon = cutil.add_cyclic_point(
            lag_longitude_correlation[variable.name],
            coord=lag_longitude_correlation[variable.name].lon
        )

        levels = np.linspace(-1, 1, 21)

        im = ax.contourf(
            clon,
            lag_longitude_correlation[variable.name].lag,
            cdata,
            cmap=mjo.modified_colormap('coolwarm', 'white', 0.05, 0.05),
            levels=levels[levels != 0]
        )
        cbar = fig.colorbar(im)
        cbar.set_ticks(np.arange(-1, 1.2, 0.2))

        ax.contour(
            clon,
            lag_longitude_correlation[variable.name].lag,
            cdata,
            colors="k",
            levels=levels[levels != 0]
        )

        # ax.invert_yaxis()
        ax.set_ylim(-30, 30)
        ax.set_xlim(0, 360 - 2.5)
        xtick_locations = np.arange(0, 360 + 60, 60)
        ax.set_xticks(ticks=xtick_locations, labels=tick_labeller(xtick_locations - 180, "lon"))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=7, prune="lower"))
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Lag (Day)")

        for spine in ax.spines.values():
            spine.set_edgecolor("#bcbcbc")
            spine.set_linewidth(3)

        # plt.show()
        if not savefig:
            plt.show()
        else:
            save_string = (
                f"{experiment}"
                + f"_{compositing_variable.attrs['file_id']}"
                + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
                + f"_PC-{PCs_to_use[0]+1}-{PCs_to_use[1]+1}_lag-longitude-correlation.png"
            )
            print(f"Saving plot as {save_string}")
            plt.savefig(
                f"{output_directory}/correlations/lag-longitude/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )

print(f"{'='*40}")
print("Finished")

## Lag-Latitude

In [ ]:
calculate_corr = True
plot_corr = True
savefig = False

lag_days = np.arange(-30, 31)
reference_lon = 180
meridional_mean_region = slice(-10, 10)
weights = np.cos(np.deg2rad(latitude))
weights.name = "weights"

variables_to_plot = [
    variables_filtered['precipitation'.title()],
    variables_filtered['outgoing longwave radiation'.title()],
    variables_filtered['zonal wind'.title()].sel(plev=200),
    variables_filtered['zonal wind'.title()].sel(plev=850),
    variables_filtered['vertical wind'.title()].sel(plev=500),
    variables_filtered['column temperature'.title()],
    variables_filtered['column water vapor'.title()],
]

if calculate_corr or 'lag_latitude_correlation' not in globals():
    lag_latitude_correlation = {}

for variable in variables_to_plot:
    print(f"Variable: {variable.name}")

    if calculate_corr:
        # lag_latitude_correlation[variable.name] = np.empty((len(lag_days), len(longitude)))
        lag_latitude_correlation[variable.name] = xr.DataArray(
            data=np.empty((len(lag_days), len(latitude))),
            dims=["lag", "lat"],
            coords=dict(
                lag=lag_days,
                lat=latitude
            )
        )

        # reference_time_series = variable.sel(lon=reference_lon, lat=meridional_mean_region).weighted(weights).mean(dim='lat')
        reference_time_series = PC[0]

        print("Computing correlations...")
        for lag_index, lag in enumerate(lag_days):
            lag_latitude_correlation[variable.name][lag_index] = np.einsum(
                    'ij,i->j',
                    standardize_time_series(variable.sel(lon=reference_lon)),
                    np.roll(standardize_time_series(reference_time_series), shift=lag),
                ) / len(reference_time_series)
        print("Correlations computed")

    if plot_corr:
        plt.style.use("bmh")
        plt.rcParams.update({"font.size": 20})
        [fig, ax] = plt.subplots(figsize=(16, 6))

        ax.set_title(
            f"{variable.name} Lag-Latitude Correlation",
            pad=10
        )

        levels = np.linspace(-1, 1, 21)

        im = ax.contourf(
            lag_latitude_correlation[variable.name].lat,
            lag_latitude_correlation[variable.name].lag,
            lag_latitude_correlation[variable.name],
            cmap=mjo.modified_colormap('coolwarm', 'white', 0.05, 0.05),
            levels=levels[levels != 0]
        )
        cbar = fig.colorbar(im)
        cbar.set_ticks(np.arange(-1, 1.2, 0.2))

        ax.contour(
            lag_latitude_correlation[variable.name].lat,
            lag_latitude_correlation[variable.name].lag,
            lag_latitude_correlation[variable.name],
            colors="k",
            levels=levels[levels != 0]
        )


        ax.set_xlim(-30, 30)
        ax.set_ylim(-30, 30)
        xtick_locations = np.arange(-30, 30 + 10, 10)
        ax.set_xticks(ticks=xtick_locations, labels=tick_labeller(xtick_locations, "lat"))
        # ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=7, prune="lower"))
        ax.set_xlabel("Latitude")
        ax.set_ylabel("Lag (Day)")

        for spine in ax.spines.values():
            spine.set_edgecolor("#bcbcbc")
            spine.set_linewidth(3)

        # plt.show()
        if not savefig:
            plt.show()
        else:
            save_string = (
                f"{experiment}"
                + f"_{compositing_variable.attrs['file_id']}"
                + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
                + f"_PC-{PCs_to_use[0]+1}-{PCs_to_use[1]+1}_lag-longitude-correlation.png"
            )
            print(f"Saving plot as {save_string}")
            plt.savefig(
                f"{output_directory}/correlations/lag-latitude/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )

print(f"{'='*40}")
print("Finished")

## Lag-Height

In [ ]:
calculate_corr = True
plot_corr = True
savefig = False

lag_days = np.arange(-30, 31)
reference_lon = 180
meridional_mean_region = slice(-10, 10)
weights = np.cos(np.deg2rad(latitude))
weights.name = "weights"

variables_to_plot = [
    variables_filtered['zonal wind'.title()],
    variables_filtered['meridional wind'.title()],
    variables_filtered['vertical wind'.title()],
    variables_filtered['temperature'.title()],
    variables_filtered['moisture'.title()],
    variables_filtered['geopotential height'.title()],
    variables_filtered['moist static energy'.title()],
    variables_filtered['longwave heating rate'.title()],
    variables_filtered['shortwave heating rate'.title()],
]

if calculate_corr or 'lag_height_correlation' not in globals():
    lag_height_correlation = {}

for variable in variables_to_plot:
    print(f"Variable: {variable.name}")

    if calculate_corr:
        lag_height_correlation[variable.name] = xr.DataArray(
            data=np.empty((len(lag_days), len(pressure_levels))),
            dims=["lag", "plev"],
            coords=dict(
                lag=lag_days,
                plev=pressure_levels
            )
        )

        # reference_time_series = variable.sel(lon=reference_lon, lat=meridional_mean_region, plev=500).weighted(weights).mean(dim='lat')
        reference_time_series = variables_filtered['Precipitation'].sel(
            lon=reference_lon, lat=meridional_mean_region
        ).weighted(weights).mean(dim='lat')
        # reference_time_series = PC[0]

        print("Computing correlations...")
        for lag_index, lag in enumerate(lag_days):
            lag_height_correlation[variable.name][lag_index] = np.einsum(
                    'ij,i->j',
                    # standardize_time_series(variable.sel(lon=reference_lon, lat=meridional_mean_region).weighted(weights).mean(dim='lat')),
                    variable.sel(lon=reference_lon, lat=meridional_mean_region).weighted(weights).mean(dim='lat'),
                    np.roll(standardize_time_series(reference_time_series), shift=lag),
                ) / len(reference_time_series)
        print("Correlations computed")

    if plot_corr:
        plt.style.use("bmh")
        plt.rcParams.update({"font.size": 20})
        [fig, ax] = plt.subplots(figsize=(16, 6))

        ax.set_title(
            f"{variable.name} Lag-Height Correlation",
            pad=10
        )

        # levels = np.linspace(-1, 1, 21)
        levels = np.linspace(
            lag_height_correlation[variable.name].min(),
            lag_height_correlation[variable.name].max(),
            21
        )

        im = ax.contourf(
            lag_height_correlation[variable.name].lag,
            lag_height_correlation[variable.name].plev,
            lag_height_correlation[variable.name].T,
            cmap=mjo.modified_colormap('coolwarm', 'white', 0.05, 0.05),
            norm=mcolors.CenteredNorm(vcenter=0),
            levels=levels[levels != 0]
        )
        cbar = fig.colorbar(im)
        cbar.set_label(f"{variable.attrs['units']}")
        # cbar.set_ticks(np.arange(-1, 1.2, 0.2))

        # ax.contour(
        #     lag_height_correlation[variable.name].lag,
        #     lag_height_correlation[variable.name].plev,
        #     lag_height_correlation[variable.name].T,
        #     colors="k",
        #     levels=levels[levels != 0]
        # )

        ax.set_xlim(-30, 30)
        ax.set_xticks(np.arange(-30, 35, 5))
        ax.invert_xaxis()
        # ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=7, prune="lower"))
        ax.set_xlabel("Lag (Day)")

        ax.set_ylim(100, 950)
        ax.set_yticks(np.arange(100, 1000, 100))
        ax.set_ylabel("Pressure (hPa)")
        ax.invert_yaxis()

        for spine in ax.spines.values():
            spine.set_edgecolor("#bcbcbc")
            spine.set_linewidth(3)

        # plt.show()
        if not savefig:
            plt.show()
        else:
            save_string = (
                f"{experiment}"
                + f"_{compositing_variable.attrs['file_id']}"
                + f"{((str(compositing_variable.plev.values)) if 'plev' in compositing_variable.coords else '')}"
                + f"_PC-{PCs_to_use[0]+1}-{PCs_to_use[1]+1}_lag-longitude-correlation.png"
            )
            print(f"Saving plot as {save_string}")
            plt.savefig(
                f"{output_directory}/correlations/lag-longitude/{save_string}",
                dpi=500,
                bbox_inches="tight",
            )

print(f"{'='*40}")
print("Finished")

# Lag Composites

## Lag-Height

In [ ]:
print("Compositing over chosen events")
print(f"{'='*config.SEP_WIDTH}")

variables_to_plot = [
    variables_filtered['precipitation'.title()],
    variables_filtered['zonal wind'.title()],
    variables_filtered['meridional wind'.title()],
    variables_filtered['vertical wind'.title()],
    variables_filtered['temperature'.title()],
    variables_filtered['moisture'.title()],
    variables_filtered['geopotential height'.title()],
    variables_filtered['moist static energy'.title()],
    variables_filtered['longwave heating rate'.title()],
    variables_filtered['shortwave heating rate'.title()],
]


variables_composited = {}
composite_days = np.arange(-15, 16)

meridional_mean_reference_precip = variables_filtered['Precipitation'].sel(
    lon=reference_lon,
    lat=meridional_mean_region
).weighted(weights).mean(dim='lat').isel(time=slice(15, len(time)-15))

precip_std = meridional_mean_reference_precip.std(dim='time')
precipitation_peaks = meridional_mean_reference_precip.where(meridional_mean_reference_precip >= precip_std).dropna(dim='time')

precipitation_peaks = precipitation_peaks.drop_sel(
        time=[
            cftime.DatetimeNoLeap(7, 2, 16, 0, 0, 0, 0, has_year_zero=True),
            cftime.DatetimeNoLeap(7, 2, 17, 0, 0, 0, 0, has_year_zero=True),
            cftime.DatetimeNoLeap(7, 2, 18, 0, 0, 0, 0, has_year_zero=True),
            cftime.DatetimeNoLeap(7, 2, 19, 0, 0, 0, 0, has_year_zero=True),
            cftime.DatetimeNoLeap(7, 2, 19, 0, 0, 0, 0, has_year_zero=True),
            cftime.DatetimeNoLeap(7, 2, 20, 0, 0, 0, 0, has_year_zero=True),
            cftime.DatetimeNoLeap(7, 2, 21, 0, 0, 0, 0, has_year_zero=True),
            cftime.DatetimeNoLeap(7, 2, 22, 0, 0, 0, 0, has_year_zero=True),
            cftime.DatetimeNoLeap(7, 2, 23, 0, 0, 0, 0, has_year_zero=True),
            cftime.DatetimeNoLeap(7, 2, 24, 0, 0, 0, 0, has_year_zero=True)
        ]
    )

for index, variable in enumerate(variables_to_plot):
# for index, variable in enumerate([variables_filtered['precipitation'.title()]]):
    print(f"{f'({index+1}/{len(variables_to_plot)}) {variable.name}...':<{config.SEP_WIDTH-1}}", end="")
    meridional_mean_reference_variable = variables_filtered[variable.name].sel(
                lon=reference_lon,
                lat=meridional_mean_region
            ).weighted(weights).mean(dim='lat')

    if 'plev' in variable.coords:
        variables_composited[variable.name] = xr.DataArray(
            data = np.empty((len(composite_days), len(pressure_levels))),
            dims=["day", "plev"],
            coords = dict(
                day=composite_days,
                plev=pressure_levels
            ),
            attrs=variables_filtered[variable.name].attrs,
            name=variable.name
        )
    else:
        variables_composited[variable.name] = xr.DataArray(
            data = np.empty((len(composite_days))),
            dims=["day"],
            coords = dict(
                day=composite_days,
            ),
            attrs=variables_filtered[variable.name].attrs,
            name=variable.name
        )

    for day_index, composite_day in enumerate(composite_days):
        variables_composited[variable.name][day_index] = np.average(
            [
                meridional_mean_reference_variable.sel(time=(peak_day + timedelta(days=int(composite_day))).values)
                for peak_day in precipitation_peaks.time
            ],
            axis=0
        )

    print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

In [ ]:
for variable in variables_composited.values():
    plt.style.use("default")
    plt.rcParams.update({"font.size": 24})

    if variable.name != 'Precipitation':
        fig = plt.figure(figsize=(16, 9))
        gs = GridSpec(2, 1, height_ratios=[30, 2], figure=fig)
        gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.32, wspace=0.0)

        ax = fig.add_subplot(gs[0])
        cb_ax = fig.add_subplot(gs[1])

        ax.set_title(
            f"{experiment} {variable.name} composited relative to\n PRCP events >= 1 standard deviation",
            pad=15
        )
        # Plot data

        im = ax.contourf(
            variable.day,
            variable.plev,
            variable.T,
            levels=16,
            norm=mcolors.CenteredNorm(vcenter=0),
            **{k: copy.deepcopy(v) for k, v in {
                "cmap": plotting_attributes[variable.name].get("d_cmap")
            }.items() if v is not None}
        )

        # Add colorbar
        cbar = fig.colorbar(
            im,
            cax=cb_ax,
            label=variable.attrs['units'],
            orientation="horizontal",
        )
        cbar.ax.tick_params(labelsize=20)

    ax2 = ax.twinx()
    ax2.plot(
        variables_composited['Precipitation'].day,
        variables_composited['Precipitation'],
        color='k',
        lw=3
    )

    # Axis parameters
    ax.set_xlim(-15, 15)
    x_ticks = np.arange(-15, 15+3, 3)
    ax.set_xticks(x_ticks)
    # ax.xaxis.set_major_locator(mticker.MaxNLocator(7, prune="lower"))
    ax.set_xlabel("Lag (day)")
    ax.invert_xaxis()

    # ax.set_yscale('log')
    ax.set_ylim(100, 950)
    ax.set_yticks(np.arange(100, 1000, 100))
    ax.set_ylabel("Pressure (hPa)")
    ax.invert_yaxis()

    plt.show()
print("Finished")

# Hovmoller

In [ ]:
variable = "outgoing longwave radiation"

plt.style.use("default")
plt.rcParams.update({"font.size": 24})

# Create figure and gridpsec
fig = plt.figure(figsize=(16, 9))
gs = GridSpec(1, 2, width_ratios=[100, 2])
gs.update(left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.05)

# Specify axes
ax = fig.add_subplot(gs[0])
cbar_ax = fig.add_subplot(gs[1])

# Label plot
ax.set_title(
    f"Hovmoller of Intraseasonally-filtered {experiment} {variables_filtered[variable].name}",
    fontsize=24,
    pad=12,
)

# Plot data
im = ax.contourf(
    longitude,
    np.arange(len(time))[:150],
    variables_filtered["outgoing longwave radiation"].sel(lat=slice(-10, 10)).mean(dim="lat")[:150],
    cmap=mjo.modified_colormap("coolwarm", "white", 0.05, 0.5),
    norm=mcolors.CenteredNorm(),
    levels=21
)

# Add colorbar
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=20)
cbar.set_label(variables_filtered[variable].attrs["units"])

# Add line at 180°
ax.axvline(x=180, color="gray", lw=3, ls="--", alpha=0.5)

# Configure axes
ax.set_xticks(np.arange(0, 360, 30))
ax.set_aspect("auto")
ax.set_xlabel("Longitude")
ax.set_ylabel("Time (days)")

plt.show()
# plt.savefig(
#     f"{output_directory}/hovmoller_intraseasonally_filtered_precipitation_30S-30N.png",
#     dpi=300,
#     bbox_inches='tight'
# )

# Space-time Decompositions

## Power Spectra

### Calculate space-time power spectra

In [ ]:
print("Calculate space-time spectra")
print(f"{'':{'='}^{30}}")

max_latitude = 33
mask_land = False

plot_raw_spectrum = False
plot_background_spectrum = True
plot_signal_strength = True
save_plots = True
n_smooths = 15

# processed_data = variables_detrended["outgoing longwave radiation"]
processed_data = variables_detrended["Precipitation"]

print("   → Computing symmetric and asymmetric signals...")
# Separate into symmetric/antisymmetric component
symmetric_signal = xr.zeros_like(processed_data.sel(lat=slice(0, max_latitude)))
asymmetric_signal = xr.zeros_like(processed_data.sel(lat=slice(0, max_latitude)))

for latitude_index, latitude in enumerate(symmetric_signal.lat):
    symmetric_signal[:, latitude_index, :] = (1 / 2) * (
        processed_data.sel(lat=latitude, method="nearest")
        + processed_data.sel(lat=-latitude, method="nearest")
    )
    asymmetric_signal[:, latitude_index, :] = -(1 / 2) * (
        processed_data.sel(lat=latitude, method="nearest")
        - processed_data.sel(lat=-latitude, method="nearest")
    )

symmetric_signal = symmetric_signal.fillna(0)
asymmetric_signal = asymmetric_signal.fillna(0)

# Subset into segments in time (96 days, overlap 60 days)
segment_length = 192
overlap = 192//2
window_width = 5

# segment_length = 256
# overlap = 256//2
# window_width = 5

# Define the Hann window
hann_window = np.concatenate(
    (
        np.hanning(window_width),
        np.ones(segment_length - window_width * 2),
        np.hanning(window_width),
    ),
    axis=0,
)

print("   → Segmenting data...")
# Segment the data
symmetric_signal_segmented = symmetric_signal.rolling(
    time=segment_length, center=False
).construct("segment_index")

symmetric_signal_segmented = symmetric_signal_segmented[(segment_length - 1) :][
    :: (segment_length - overlap)
].transpose("time", "segment_index", "lat", "lon")

asymmetric_signal_segmented = asymmetric_signal.rolling(
    time=segment_length, center=False
).construct("segment_index")

asymmetric_signal_segmented = asymmetric_signal_segmented[(segment_length - 1) :][
    :: (segment_length - overlap)
].transpose("time", "segment_index", "lat", "lon")

print("   → Detrending segmented data...")
# Detrend the data along the segmented axis
symmetric_signal_detrended = symmetric_signal_segmented.copy(deep=True)
asymmetric_signal_detrended = asymmetric_signal_segmented.copy(deep=True)
symmetric_signal_detrended.values = signal.detrend(symmetric_signal_segmented, axis=1)
asymmetric_signal_detrended.values = signal.detrend(asymmetric_signal_segmented, axis=1)

print("   → Windowing segmented data...")
# Apply the Hann window to each segment
symmetric_signal_windowed = symmetric_signal_detrended.copy(deep=True)
asymmetric_signal_windowed = asymmetric_signal_detrended.copy(deep=True)
symmetric_signal_windowed.values = np.einsum(
    "j,ijkl->ijkl", hann_window, symmetric_signal_detrended
)
asymmetric_signal_windowed.values = np.einsum(
    "j,ijkl->ijkl", hann_window, asymmetric_signal_detrended
)

print("   → Fourier transforming segmented data...")
# Fourier transform the data and calculate raw power
symmetric_signal_fft = (
    np.fft.fft2(symmetric_signal_windowed, axes=(1, 3))
    / (len(longitude) * segment_length)
    * 4
)
raw_symmetric_power = np.fft.fftshift(
    np.mean(np.real(symmetric_signal_fft * np.conj(symmetric_signal_fft)), axis=(0, 2))
)

asymmetric_signal_fft = (
    np.fft.fft2(asymmetric_signal_windowed, axes=(1, 3))
    / (len(longitude) * segment_length)
    * 4
)
raw_asymmetric_power = np.fft.fftshift(
    np.mean(
        np.real(asymmetric_signal_fft * np.conj(asymmetric_signal_fft)), axis=(0, 2)
    )
)

# Calculate the frequency and zonal wavenumber axis coordinates
frequency = np.arange(-segment_length / 2, segment_length / 2) * 1 / segment_length
zonal_wavenumber = (
    np.arange(-len(longitude) / 2, len(longitude) / 2)
    * (1 / 2.5)
    / len(longitude)
    * 360
)
# ZONAL_, y = np.meshgrid(zonal_wavenumber, -frequency)

#### 1-2-1 Filtering
print("   → 1-2-1 filtering background...")
# Smooths the background spectrum 'n_smooths' times
background_spectrum = (raw_symmetric_power + raw_asymmetric_power) / 2
background_spectrum = one_two_one_filter(background_spectrum, n_smooths, "time")
background_spectrum = one_two_one_filter(background_spectrum, n_smooths, "space")

# Calculate signal strength as raw/smoothed background
symmetric_power_spectrum = raw_symmetric_power / background_spectrum
asymmetric_power_spectrum = raw_asymmetric_power / background_spectrum

print(f"{'':{'='}^{30}}")
print("Space-time spectra calculated")

In [ ]:
print(f"{'Space-Time Power Spectra':^{config.SEP_WIDTH}}")

max_latitude = 33
mask_land = False

plot_raw_spectrum = False
plot_background_spectrum = True
plot_signal_strength = True
save_plots = True
n_smooths = 15

# processed_data = variables_detrended["outgoing longwave radiation"]
# processed_data = variables_detrended["precipitation"]

symmetric_signal_fft = {}
asymmetric_signal_fft = {}
raw_symmetric_power = {}
raw_asymmetric_power = {}
background_spectrum = {}
symmetric_power_spectrum = {}
asymmetric_power_spectrum = {}

for index, variable in enumerate([
    variables_detrended['Precipitation'],
    variables_detrended['Outgoing Longwave Radiation'],
    variables_detrended['Zonal Wind'].sel(plev=850),
    variables_detrended['Zonal Wind'].sel(plev=200),
    variables_detrended['Vertical Wind'].sel(plev=500)
]):

    variable_id = f"{variable.attrs['file_id']}{(str(variable.plev.values) if 'plev' in variable.coords else '')}"
    print(f"{'='*config.SEP_WIDTH}")
    print(f"{f'Variable: {variable_id}':^{config.SEP_WIDTH}}")
    print(f"{'='*config.SEP_WIDTH}")
    print("   → Computing symmetric and asymmetric signals...")
    # Separate into symmetric/antisymmetric component
    symmetric_signal = xr.zeros_like(variable.sel(lat=slice(0, max_latitude)))
    asymmetric_signal = xr.zeros_like(variable.sel(lat=slice(0, max_latitude)))

    for latitude_index, latitude in enumerate(symmetric_signal.lat):
        symmetric_signal[:, latitude_index, :] = (1 / 2) * (
            variable.sel(lat=latitude, method="nearest")
            + variable.sel(lat=-latitude, method="nearest")
        )
        asymmetric_signal[:, latitude_index, :] = -(1 / 2) * (
            variable.sel(lat=latitude, method="nearest")
            - variable.sel(lat=-latitude, method="nearest")
        )

    symmetric_signal = symmetric_signal.fillna(0)
    asymmetric_signal = asymmetric_signal.fillna(0)

    # Subset into segments in time (96 days, overlap 60 days)
    segment_length = 192
    overlap = 192//2
    window_width = 5

    # Define the Hann window
    hann_window = np.concatenate(
        (
            np.hanning(window_width),
            np.ones(segment_length - window_width * 2),
            np.hanning(window_width),
        ),
        axis=0,
    )

    print("   → Segmenting data...")
    # Segment the data
    symmetric_signal_segmented = symmetric_signal.rolling(
        time=segment_length, center=False
    ).construct("segment_index")

    symmetric_signal_segmented = symmetric_signal_segmented[(segment_length - 1) :][
        :: (segment_length - overlap)
    ].transpose("time", "segment_index", "lat", "lon")

    asymmetric_signal_segmented = asymmetric_signal.rolling(
        time=segment_length, center=False
    ).construct("segment_index")

    asymmetric_signal_segmented = asymmetric_signal_segmented[(segment_length - 1) :][
        :: (segment_length - overlap)
    ].transpose("time", "segment_index", "lat", "lon")

    print("   → Detrending segmented data...")
    # Detrend the data along the segmented axis
    symmetric_signal_detrended = xr.zeros_like((symmetric_signal_segmented))
    asymmetric_signal_detrended = xr.zeros_like((asymmetric_signal_segmented))
    symmetric_signal_detrended.values = signal.detrend(symmetric_signal_segmented, axis=1)
    asymmetric_signal_detrended.values = signal.detrend(asymmetric_signal_segmented, axis=1)

    print("   → Windowing segmented data...")
    # Apply the Hann window to each segment
    symmetric_signal_windowed = xr.zeros_like((symmetric_signal_detrended))
    asymmetric_signal_windowed = xr.zeros_like((asymmetric_signal_detrended))
    symmetric_signal_windowed.values = np.einsum(
        "j,ijkl->ijkl", hann_window, symmetric_signal_detrended
    )
    asymmetric_signal_windowed.values = np.einsum(
        "j,ijkl->ijkl", hann_window, asymmetric_signal_detrended
    )

    print("   → Fourier transforming segmented data...")
    # Fourier transform the data and calculate raw power
    symmetric_signal_fft[variable_id] = (
        np.fft.fft2(symmetric_signal_windowed, axes=(1, 3))
        / (len(longitude) * segment_length)
        * 4
    )
    raw_symmetric_power[variable_id] = np.fft.fftshift(
        np.mean(np.real(symmetric_signal_fft[variable_id] * np.conj(symmetric_signal_fft[variable_id])), axis=(0, 2))
    )

    asymmetric_signal_fft[variable_id] = (
        np.fft.fft2(asymmetric_signal_windowed, axes=(1, 3))
        / (len(longitude) * segment_length)
        * 4
    )
    raw_asymmetric_power[variable_id] = np.fft.fftshift(
        np.mean(
            np.real(asymmetric_signal_fft[variable_id] * np.conj(asymmetric_signal_fft[variable_id])), axis=(0, 2)
        )
    )

    # Calculate the frequency and zonal wavenumber axis coordinates
    frequency = np.arange(-segment_length / 2, segment_length / 2) * 1 / segment_length
    zonal_wavenumber = (
        np.arange(-len(longitude) / 2, len(longitude) / 2)
        * (1 / 2.5)
        / len(longitude)
        * 360
    )
    # ZONAL_, y = np.meshgrid(zonal_wavenumber, -frequency)

    #### 1-2-1 Filtering
    print("   → 1-2-1 filtering background...")
    # Smooths the background spectrum 'n_smooths' times
    background_spectrum[variable_id] = (raw_symmetric_power[variable_id] + raw_asymmetric_power[variable_id]) / 2
    background_spectrum[variable_id] = one_two_one_filter(background_spectrum[variable_id], n_smooths, "time")
    background_spectrum[variable_id] = one_two_one_filter(background_spectrum[variable_id], n_smooths, "space")

    # Calculate signal strength as raw/smoothed background
    symmetric_power_spectrum[variable_id] = raw_symmetric_power[variable_id] / background_spectrum[variable_id]
    asymmetric_power_spectrum[variable_id] = raw_asymmetric_power[variable_id] / background_spectrum[variable_id]

print(f"{'':{'='}^{config.SEP_WIDTH}}")
print("Finished")

### Plot space-time power spectra

In [ ]:
savefig = False

plt.style.use("bmh")
plt.rcParams.update({"font.size": 16})

fig = plt.figure(figsize=(9, 9))
gs = GridSpec(1, 2, width_ratios=[1, 0.025], figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, wspace=0.1)

ax = fig.add_subplot(gs[0])
cbar_ax = fig.add_subplot(gs[1])

variable_to_plot = 'ω500'

ax.set_title(
    f"{experiment} {variable_to_plot}\n Symmetric Space-Time Power Spectrum",
    pad=15
)

im = ax.contourf(
    zonal_wavenumber,
    -frequency,
    np.log10(symmetric_power_spectrum[variable_to_plot]),
    norm=mcolors.CenteredNorm(vcenter=0),
    cmap=mjo.modified_colormap("coolwarm", "white", 0.025, 0.5),
    # levels=np.linspace(-1.2, 0.9, 21),
    # levels=np.arange(-1.2, 0.91, 0.1),
    # levels=np.linspace(-1.8, 0.8, 21),
    levels=31,
    # extend='both'
)

# divider = make_axes_locatable(ax)
# cbar_ax = divider.append_axes("right", size="2.5%", pad=0.25)

cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_ticks(
    # np.arange(-1.2, 0.91, 0.2)
    im.levels[::2]
)

# Mark 3, 6, 20 day period:
plot_days = [3, 6, 20, 100]
for day in plot_days:
    ax.axhline(y=1 / day, color="k", lw=1, ls=":")
    ax.text(-14.8, 1 / day + 0.005, str(day) + "d", fontsize=15)

# Mark zonal wavenumber == 0:
ax.axvline(x=0, color="k", lw=1, ls=":")

ax.set_xlim(-15, 15)
ax.set_xlabel("Zonal wavenumber")
ax.set_ylim(1 / 180, 1 / 2)
ax.set_ylabel("Frequency")
ax.set_aspect(30 / (1 / 2 - 1 / 180))

if not savefig:
    plt.show()
else:
    save_string = (
        f"symmetric_{experiment}"
      + f"_{variable_to_plot}"
      + f"_space-time-power-spectrum.png"
        )
    print(f"Saving plot as {save_string}")
    plt.savefig(
        f"{output_directory}/space-time-power-spectra/{save_string}",
        dpi=500,
        bbox_inches="tight",
    )

print(f"{'='*40}")
print("Finished")

In [ ]:
savefig = True

plt.style.use("bmh")
plt.rcParams.update({"font.size": 16})

fig = plt.figure(figsize=(9, 9))
gs = GridSpec(1, 2, width_ratios=[1, 0.025], figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, wspace=0.1)

ax = fig.add_subplot(gs[0])
cbar_ax = fig.add_subplot(gs[1])

ax.set_title(
    f"{experiment} {processed_data.name}\n Asymmetric Space-Time Power Spectrum",
    pad=15
)

im = ax.contourf(
    zonal_wavenumber,
    -frequency,
    np.log10(asymmetric_power_spectrum),
    norm=mcolors.CenteredNorm(vcenter=0),
    cmap=mjo.modified_colormap("coolwarm", "white", 0.05, 0.5),
    # levels=np.linspace(-1.2, 0.9, 21),
    levels=np.arange(-1.2, 0.91, 0.1),
)

# divider = make_axes_locatable(ax)
# cbar_ax = divider.append_axes("right", size="2.5%", pad=0.25)

cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_ticks(np.arange(-1.2, 0.91, 0.2))

# Mark 3, 6, 20 day period:
plot_days = [3, 6, 20, 100]
for day in plot_days:
    ax.axhline(y=1 / day, color="k", lw=1, ls=":")
    ax.text(-14.8, 1 / day + 0.005, str(day) + "d", fontsize=15)

# Mark zonal wavenumber == 0:
ax.axvline(x=0, color="k", lw=1, ls=":")

ax.set_xlim(-15, 15)
ax.set_xlabel("Zonal wavenumber")
ax.set_ylim(1 / 180, 1 / 2)
ax.set_ylabel("Frequency")
ax.set_aspect(30 / (1 / 2 - 1 / 180))

if not savefig:
    plt.show()
else:
    save_string = (
        f"asymmetric_{experiment}"
      + f"_{processed_data.attrs['file_id']}"
      + f"{((str(processed_data.plev.values)) if 'plev' in processed_data.coords else '')}"
      + f"_space-time-power-spectrum.png"
        )
    print(f"Saving plot as {save_string}")
    plt.savefig(
        f"{output_directory}/space-time-power-spectra/{save_string}",
        dpi=500,
        bbox_inches="tight",
    )

print(f"{'='*40}")
print("Finished")

## Coherence

### Calculate coherence

In [ ]:
coherence_var1 = 'PRCP'
coherence_var2 = 'U850'

cross_spectrum = np.fft.fftshift(
        np.mean(symmetric_signal_fft[coherence_var1] * np.conj(symmetric_signal_fft[coherence_var2]), axis=(0, 2))
    )

power_coh1 = np.fft.fftshift(
        np.mean(symmetric_signal_fft[coherence_var1] * np.conj(symmetric_signal_fft[coherence_var1]), axis=(0, 2))
    )

power_coh2 = np.fft.fftshift(
        np.mean(symmetric_signal_fft[coherence_var2] * np.conj(symmetric_signal_fft[coherence_var2]), axis=(0, 2))
    )

coherence = np.real(np.abs(cross_spectrum)**2/(power_coh1*power_coh2))

x_phase = np.imag(cross_spectrum)/np.abs(cross_spectrum)
x_phase[coherence <= 0.03] = np.nan
y_phase = np.real(cross_spectrum)/np.abs(cross_spectrum)
y_phase[coherence <= 0.03] = np.nan

print("Finished")

In [ ]:
savefig = True

plt.style.use("bmh")
plt.rcParams.update({"font.size": 16})

fig = plt.figure(figsize=(9, 9))
gs = GridSpec(1, 2, width_ratios=[1, 0.025], figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, wspace=0.1)

ax = fig.add_subplot(gs[0])
cbar_ax = fig.add_subplot(gs[1])

ax.set_facecolor('white')
ax.set_title(
    rf"{experiment} Coherence$^{2}$ between {coherence_var1} & {coherence_var2}",
    pad=10
)

im = ax.contourf(
    zonal_wavenumber,
    -frequency,
    coherence,
    levels=np.arange(0.1, 1, 0.05),
    # norm=mcolors.CenteredNorm(vcenter=0),
    # cmap=mjo.modified_colormap("coolwarm", "white", 0.05, 0.5),
    cmap='YlOrRd',
    # extend='both'
)

# divider = make_axes_locatable(ax)
# cbar_ax = divider.append_axes("right", size="2.5%", pad=0.25)

cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_ticks(
    # np.arange(-1.2, 0.91, 0.2)
    im.levels[::2]
)
# cbar.set_title(r"coh$^{2}$")

ax.quiver(
    zonal_wavenumber[::2],
    -frequency[::4],
    x_phase[::4, ::2],
    y_phase[::4, ::2],
    scale=15
)


# Mark 3, 6, 20 day period:
plot_days = [3, 6, 20, 100]
for day in plot_days:
    ax.axhline(y=1 / day, color="k", lw=1, ls=":")
    ax.text(-14.8, 1 / day + 0.005, str(day) + "d", fontsize=15)

# Mark zonal wavenumber == 0:
ax.axvline(x=0, color="k", lw=1, ls=":")

ax.set_xlim(-15, 15)
ax.set_xlabel("Zonal wavenumber")
ax.set_ylim(1 / 180, 1 / 2)
ax.set_ylabel("Frequency")
ax.set_aspect(30 / (1 / 2 - 1 / 180))

if not savefig:
    plt.show()
else:
    save_string = (
        f"symmetric_{experiment}"
      + f"_{coherence_var1}-{coherence_var2}_coherence.png"
        )
    print(f"Saving plot as {save_string}")
    plt.savefig(
        f"{output_directory}/coherence/{save_string}",
        dpi=500,
        bbox_inches="tight",
    )

print(f"{'='*40}")
print("Finished")

# MSE Budget

### Calculate budget terms

In [ ]:
latitude

In [ ]:
print(f"{f'Budget Terms':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

# Calculate x and y distance grids
METERS_PER_DEGREE = 111.320e3
zonal_distance = np.einsum(
    'i,j->ij',
    METERS_PER_DEGREE*np.cos(np.deg2rad(latitude.values)),
    longitude.values
)
meridional_distance = METERS_PER_DEGREE*latitude.values

# Column MSE
print(f"{'Calculating column MSE...':<{config.SEP_WIDTH-1}}", end="")
column_MSE = (100/9.81)*variables_subset['Moist Static Energy'].sel(plev=slice(100,950)).integrate('plev')
column_MSE.attrs['file_id'] = r"$\langle m \rangle$"
column_MSE.name = 'Column Moist Static Energy'
meridional_mean_column_MSE = column_MSE.sel(lat=slice(-10, 10)).mean(dim='lat')
print(rf"{'✔':>1}")

# Calculate MSE gradients
zonal_MSE_gradient = xr.zeros_like((variables_subset['Moist Static Energy']))

print(f"{'Calculating zonal MSE gradient...':<{config.SEP_WIDTH-1}}", end="")
for lat_index in range(len(latitude)):
    zonal_MSE_gradient[:, lat_index, :, :] = np.gradient(
        variables_subset['Moist Static Energy'][:, lat_index, :, :].values,
        zonal_distance[lat_index],
        axis=1
    )
print(rf"{'✔':>1}")

print(f"{'Calculating merid. MSE gradient...':<{config.SEP_WIDTH-1}}", end="")
meridional_MSE_gradient = xr.zeros_like((variables_subset['Moist Static Energy']))
meridional_MSE_gradient.values = np.gradient(
    variables_subset['Moist Static Energy'],
    meridional_distance,
    axis=1
)
print(rf"{'✔':>1}")

# Column MSE tendency
print(f"{'Calculating column MSE tendency...':<{config.SEP_WIDTH-1}}", end="")
column_MSE_tendency = column_MSE.differentiate(coord='time')
column_MSE_tendency.attrs['file_id'] = r"$\langle \partial_{t}m \rangle$"
column_MSE_tendency.name = 'Column Moist Static Energy Tendency'
meridional_mean_column_MSE_tendency = column_MSE_tendency.sel(lat=slice(-10, 10)).mean(dim='lat')
print(rf"{'✔':>1}")

# Vertical advection
print(f"{'Calculating vertical advection...':<{config.SEP_WIDTH-1}}", end="")
vertical_advection = variables_subset['Vertical Wind']*variables_subset['Moist Static Energy'].differentiate(coord='plev')/100
column_vertical_advection = (100/9.8)*vertical_advection.sel(plev=slice(100, 950)).integrate('plev')
column_vertical_advection.name = 'Column Vertical Advection'
column_vertical_advection.attrs['file_id'] = r"$\langle \omega \partial_{p}m \rangle$"
meridional_mean_column_vertical_advection = column_vertical_advection.sel(lat=slice(-10, 10)).mean(dim='lat', keep_attrs=True)
print(rf"{'✔':>1}")

# Zonal advection
print(f"{'Calculating zonal advection...':<{config.SEP_WIDTH-1}}", end="")
zonal_advection = variables_subset['Zonal Wind']*zonal_MSE_gradient
column_zonal_advection = (100/9.8)*zonal_advection.sel(plev=slice(100,950)).integrate(coord='plev')
column_zonal_advection.name = 'Column Zonal Advection'
column_zonal_advection.attrs['file_id'] = r"$\langle u \partial_{x}m \rangle$"
meridional_mean_column_zonal_advection = column_zonal_advection.sel(lat=slice(-10, 10)).mean(dim='lat', keep_attrs=True)
print(rf"{'✔':>1}")

# Meridional advection
print(f"{'Calculating merid. advection...':<{config.SEP_WIDTH-1}}", end="")
meridional_advection = variables_subset['Meridional Wind']*meridional_MSE_gradient
column_meridional_advection = (100/9.8)*meridional_advection.sel(plev=slice(100, 950)).integrate(coord='plev')
column_meridional_advection.name = 'Column Meridional Advection'
column_meridional_advection.attrs['file_id'] = r"$\langle v \partial_{y}m \rangle$"
meridional_mean_column_meridional_advection = column_meridional_advection.sel(lat=slice(-10, 10)).mean(dim='lat', keep_attrs=True)
print(rf"{'✔':>1}")

# Other terms
meridional_mean_latent_heat = variables_subset['Latent Heat Flux'].sel(
    lat=slice(-10,10)
).mean(dim='lat', keep_attrs=True)
meridional_mean_sensible_heat = variables_subset['Sensible Heat Flux'].sel(
    lat=slice(-10,10)
).mean(dim='lat', keep_attrs=True)
meridional_mean_longwave_heating = variables_subset['Column Longwave Heating'].sel(
    lat=slice(-10,10)
).mean(dim='lat', keep_attrs=True)
meridional_mean_shortwave_heating = variables_subset['Column Shortwave Heating'].sel(
    lat=slice(-10,10)
).mean(dim='lat', keep_attrs=True)
meridonal_mean_residual = meridional_mean_column_MSE_tendency + (
    + meridional_mean_column_vertical_advection
    + meridional_mean_column_zonal_advection
    + meridional_mean_column_meridional_advection
    - meridional_mean_latent_heat
    - meridional_mean_sensible_heat
    - meridional_mean_longwave_heating
    - meridional_mean_shortwave_heating
)

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

## Subset budget variables

In [ ]:
print("Subset budget variables")
print(f"{'='*config.SEP_WIDTH}")

budget_variables_dict = {
    'Moist Static Energy': meridional_mean_column_MSE,
    'MSE Tendency': meridional_mean_column_MSE_tendency,
    'Vertical Advection': meridional_mean_column_vertical_advection,
    'Zonal Advection': meridional_mean_column_zonal_advection,
    'Meridional Advection': meridional_mean_column_meridional_advection,
    'Latent Heating': meridional_mean_latent_heat,
    'Sensible Heating': meridional_mean_sensible_heat,
    'Longwave Heating': meridional_mean_longwave_heating,
    'Shortwave Heating': meridional_mean_shortwave_heating,
    'Residual': meridional_mean_residual
}

budget_variables_subset = {}
budget_variables_subset_first_half = {}
budget_variables_subset_second_half = {}

for index, budget_variable in enumerate(budget_variables_dict):
    print(f"{f'({index+1}/{len(budget_variables_dict)}) {budget_variable}...':<{config.SEP_WIDTH-1}}", end="")
    budget_variables_subset_first_half[budget_variable] = budget_variables_dict[budget_variable].sel(
        time=slice(START_TIME, MID_STOP),
    )

    budget_variables_subset_second_half[budget_variable] = budget_variables_dict[budget_variable].copy(deep=True)
    budget_variables_subset_second_half[budget_variable] = budget_variables_dict[budget_variable].sel(
        time=slice(MID_START, END_TIME),
    )
    print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

## Intraseasonal-filtering

In [ ]:
print(f"{'Filter budget variables':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

budget_variables_filtered = {}
budget_variables_filtered_first_half = {}
budget_variables_filtered_second_half = {}

nyq = 0.5
filter_order = 4
low = (1 / INTRASEASONAL_LOWCUT) / nyq
high = (1 / INTRASEASONAL_HIGHCUT) / nyq
b, a = signal.butter(filter_order, [low, high], btype="band")

for index, budget_variable in enumerate(budget_variables_dict):
    print(
        f"{f'({index+1}/{len(budget_variables_dict)}) {budget_variable}...':<{config.SEP_WIDTH-1}}",
        end=""
    )

    # Initialize filtered variables arrays
    budget_variables_filtered_first_half[budget_variable] = xr.zeros_like(
        (budget_variables_subset_first_half[budget_variable])
    )

    budget_variables_filtered_second_half[budget_variable] = xr.zeros_like(
        (budget_variables_subset_second_half[budget_variable])
    )

    # Filter the data
    budget_variables_filtered_first_half[budget_variable].values = signal.filtfilt(
        b, a, budget_variables_subset_first_half[budget_variable], axis=0
    )

    budget_variables_filtered_second_half[budget_variable].values = signal.filtfilt(
        b, a, budget_variables_subset_second_half[budget_variable], axis=0
    )

    # Concatenate halves together
    budget_variables_filtered[budget_variable] = xr.concat(
        (
            budget_variables_filtered_first_half[budget_variable],
            budget_variables_filtered_second_half[budget_variable]
        ),
        dim='time'
    )
    budget_variables_filtered[budget_variable].attrs["filtered"] = "True"
    budget_variables_filtered[budget_variable].attrs["file_id"] =\
        budget_variables_dict[budget_variable].attrs["file_id"]

    print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

In [ ]:
budget_variables_filtered['MSE Tendency'] + (
    + budget_variables_filtered['Vertical Advection']
    + budget_variables_filtered['Zonal Advection']
    + budget_variables_filtered['Meridional Advection']
    - budget_variables_filtered['Latent Heating']
    - budget_variables_filtered['Sensible Heating']
    - budget_variables_filtered['Longwave Heating']
    - budget_variables_filtered['Shortwave Heating']
)

## Lag Regress variables

In [ ]:
print(f"{'Lag Regressions':^{config.SEP_WIDTH}}")
print(f"{'='*config.SEP_WIDTH}")

lag_regression = {}
reference_lon = 180
# reference_variable = variables_filtered['Precipitation'].sel(lon=reference_lon, lat=slice(-10,10)).mean(dim='lat')
reference_variable = budget_variables_filtered['Moist Static Energy'].sel(lon=reference_lon)

for index, (variable_name, variable_data) in enumerate(budget_variables_filtered.items()):
    print(
        f"{f'({index+1}/{len(budget_variables_filtered)}) {variable_name}...':<{config.SEP_WIDTH-1}}",
        end=""
    )

    lag_regression[variable_name] = xr.DataArray(
        data=np.zeros((121, len(longitude))),
        dims=["lag", "lon"],
        coords=dict(
            lag=np.arange(-60,61),
            lon=longitude.values
        )
    )
    lag_regression[variable_name].attrs['file_id'] = budget_variables_dict[variable_name].attrs['file_id']

    for lag_index, lag in enumerate(lag_regression[variable_name].lag.values):
        lag_regression[variable_name][lag_index] = np.einsum(
                'ij,i->j',
                (variable_data-variable_data.mean(dim='time')),
                # np.roll(standardize_time_series(budget_variables_filtered['Moist Static Energy']), shift=lag, axis=0),
                np.roll(standardize_time_series(reference_variable), shift=lag),
            ) / len(time)

    print(rf"{'✔':>1}")

print(f"{'='*config.SEP_WIDTH}")
print("Finished")

## Plot Budget

In [ ]:
plt.style.use('bmh')

lines = {}

[fig, ax] = plt.subplots(1, 1, figsize=(16,9))

for index, (variable_name, variable_data) in enumerate(lag_regression.items()):
    if variable_name != 'Moist Static Energy':
        lines[index] = ax.plot(
            variable_data.lag,
            # variable_data.mean(dim='lon'),
            variable_data.sel(lon=reference_lon),
            label=f"{variable_data.attrs['file_id']}",
            lw=(4 if variable_name == 'MSE Tendency' else 3),
            color=('k' if variable_name == 'MSE Tendency' else bmh_colors(index-1))
        )

ax.plot(
    residual.lag,
    residual,
    color='magenta',
    label='Res.'
)

ax.set_title(
    f"MSE Budget Terms regressed on {reference_variable.attrs['file_id']} at {reference_lon}°",
    pad=15
)
ax.invert_xaxis()
ax.set_xlim(-30, 30)
ax.set_xticks(np.arange(-30, 35, 5))

ax.set_xlabel('Lag (day)')
ax.set_ylabel(r'W m$^{-2}$')
ax.axhline(y=0, color='#bcbcbc', ls=':', lw=3)

for axis in ["top", "bottom", "left", "right"]:
    ax.spines[axis].set_linewidth(4)
ax.spines['right'].set_color('k')

ax2 = ax.twinx()
ax2.plot(
    lag_regression['Moist Static Energy'].lag,
    # lag_regression['Moist Static Energy'].mean(dim='lon'),
    lag_regression['Moist Static Energy'].sel(lon=reference_lon),
    label=lag_regression['Moist Static Energy'].attrs['file_id'],
    color='k',
    ls='--',
    lw=4
)
ax2.grid(False)
ax2.invert_xaxis()
# ax2.set_xticks(np.arange(-30, 35, 5))

# ax.legend(fontsize=12, ncols=3)
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

legend_handles = [
    ([lines2[0], lines[0]], [labels2[0], labels[0]]),
    (lines[1:4], labels[1:4]),
    (lines[4:], labels[4:])
]

for i, (h, l) in enumerate(legend_handles):
    ax2.add_artist(ax2.legend(h, l, loc='upper right', bbox_to_anchor=([0.75, 0.88, 1.0][i], 1), fontsize=16))  # Adjust column positions

plt.show()

In [ ]:
plt.contour(
    lag_regression['MSE Tendency'].lon,
    lag_regression['MSE Tendency'].lag,
    lag_regression['MSE Tendency'],
    colors='k'
)

plt.contourf(
    lag_regression['Zonal Advection'].lon,
    lag_regression['Zonal Advection'].lag,
    lag_regression['Zonal Advection'] + lag_regression['Meridional Advection'],
    # colors='k'
)
plt.ylim(-30,30)

In [ ]:
residual = lag_regression['MSE Tendency'].sel(lon=reference_lon) + (
    + lag_regression['Vertical Advection'].sel(lon=reference_lon)
    + lag_regression['Zonal Advection'].sel(lon=reference_lon)
    + lag_regression['Meridional Advection'].sel(lon=reference_lon)
    - lag_regression['Latent Heating'].sel(lon=reference_lon)
    - lag_regression['Sensible Heating'].sel(lon=reference_lon)
    - lag_regression['Longwave Heating'].sel(lon=reference_lon)
    - lag_regression['Shortwave Heating'].sel(lon=reference_lon)
)